# Library dan Konfigurasi

## Import Library

In [1]:
import ast
import io
import json
import os
import re
import shutil
import subprocess
import sys
import stat
import time
import tokenize
from collections import defaultdict
from datetime import datetime
import networkx as nx
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Formatter untuk standarisasi kode Python
import autopep8
import black

# Engine export Excel untuk pandas
import openpyxl

# Progress bar & notebook display
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# Waktu
run_time = datetime.now()

# Konfigurasi visualisasi default
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

print("=" * 65)
print(f"{'LIBRARY INITIALIZATION':^65}")
print("-" * 65)
print(f"Python     : {sys.version.split()[0]}")
print(f"✅ {run_time.strftime('%Y-%m-%d %H:%M:%S')} - Libraries loaded successfully.")
print("-" * 65)

                     LIBRARY INITIALIZATION                      
-----------------------------------------------------------------
Python     : 3.10.6
✅ 2026-06-11 22:45:08 - Libraries loaded successfully.
-----------------------------------------------------------------


## Konfigurasi

In [2]:
# ==============================================================================
# DIRECTORY CONFIGURATION & INITIALIZATION
# Menentukan path utama, struktur folder dataset, dan file output
# ==============================================================================

# Root directory dan file input
BASE_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code"
DATASET_FOLDER = "dataset(2)"
OUTPUT_FOLDER = "output(2)"

INPUT_GITHUB = os.path.join(BASE_DIR, "asli", "nim_github.txt")

# Struktur folder pipeline
DIRS = {
    "DATASET"      : os.path.join(BASE_DIR, DATASET_FOLDER),

    # Preprocessing
    "RAW"          : os.path.join(BASE_DIR, DATASET_FOLDER, "00_Raw"),
    "ANON"          : os.path.join(BASE_DIR, DATASET_FOLDER, "01_Raw_Anon"),
    "NORM"         : os.path.join(BASE_DIR, DATASET_FOLDER, "02_Normalized"),
    "CONV"         : os.path.join(BASE_DIR, DATASET_FOLDER, "03_Converted"),
    "CLEAN"        : os.path.join(BASE_DIR, DATASET_FOLDER, "04_Cleaned"),

    # Formatter experiment
    "AUTOPEP8"     : os.path.join(BASE_DIR, DATASET_FOLDER, "05a_Autopep8"),
    "BLACK"        : os.path.join(BASE_DIR, DATASET_FOLDER, "05b_Black"),
    
    "FILTERED"     : os.path.join(BASE_DIR, DATASET_FOLDER, "06_Filtered"),
    "FILTERED_PRAK"     : os.path.join(BASE_DIR, DATASET_FOLDER, "06_Filtered_prak"),
    "FILTERED_TGS"     : os.path.join(BASE_DIR, DATASET_FOLDER, "06_Filtered_tgs"),

    # AST & Graph
    "AST_P"          : os.path.join(BASE_DIR, DATASET_FOLDER, "07_AST_prak"),
    "AST_T"          : os.path.join(BASE_DIR, DATASET_FOLDER, "07_AST_tgs"),
    "AST_VISUAL"   : os.path.join(BASE_DIR, DATASET_FOLDER, "07a_AST_visual"),
    "GRAPH_P"        : os.path.join(BASE_DIR, DATASET_FOLDER, "08_Graph_prak"),
    "GRAPH_T"        : os.path.join(BASE_DIR, DATASET_FOLDER, "08_Graph_tgs"),
    "INPUT_GRAPH_P"  : os.path.join(BASE_DIR, DATASET_FOLDER, "09_Graph2vec_Input_prak"),
    "INPUT_GRAPH_T"  : os.path.join(BASE_DIR, DATASET_FOLDER, "09_Graph2vec_Input_tgs"),
    "EMBEDDING_P"    : os.path.join(BASE_DIR, DATASET_FOLDER, "10_Graph2vec_Embedding_prak"),
    "EMBEDDING_T"    : os.path.join(BASE_DIR, DATASET_FOLDER, "10_Graph2vec_Embedding_tgs"),

    # Output
    "LOGS"         : os.path.join(BASE_DIR, OUTPUT_FOLDER),
    "RUNTIME_A"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_autopep8"),
    "RUNTIME_B"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_black"),
    "EVAL"         : os.path.join(BASE_DIR, OUTPUT_FOLDER, "eval"),
    "EVAL_P"       : os.path.join(BASE_DIR, OUTPUT_FOLDER, "eval", "praktikum"),
    "EVAL_T"       : os.path.join(BASE_DIR, OUTPUT_FOLDER, "eval", "tugas"),
    
    # folder output untuk grafik similarity
    "SIMILARITY"   : os.path.join(BASE_DIR, OUTPUT_FOLDER,"similarity"),
}

# File output penelitian
RESULTS = {
    # Preprocessing
    "CLONE_REPORT"    : os.path.join(DIRS["LOGS"], "01_clone_report.xlsx"),
    "NORM_REPORT"     : os.path.join(DIRS["LOGS"], "02_normalization_report.xlsx"),
    "CONV_REPORT"     : os.path.join(DIRS["LOGS"], "03_conversion_report.xlsx"),
    "CLEAN_REPORT"    : os.path.join(DIRS["LOGS"], "04_cleaning_report.xlsx"),

    # Formatter
    "ERR_AUTOPEP"     : os.path.join(DIRS["LOGS"], "05a_autopep_errors.json"),
    "ERR_BLACK"       : os.path.join(DIRS["LOGS"], "05b_black_errors.json"),

    # Runtime Autopep8
    "RUN_PROJECT_A"   : os.path.join(DIRS["RUNTIME_A"], "a_runtime_project_AUTOPEP8.xlsx"),
    "RUN_FUNCTION_A"  : os.path.join(DIRS["RUNTIME_A"], "b_runtime_function_AUTOPEP8.xlsx"),
    "RUN_COMPARE_A"   : os.path.join(DIRS["RUNTIME_A"], "c_runtime_compare_AUTOPEP8.xlsx"),

    # Runtime Black
    "RUN_PROJECT_B"   : os.path.join(DIRS["RUNTIME_B"], "a_runtime_project_BLACK.xlsx"),
    "RUN_FUNCTION_B"  : os.path.join(DIRS["RUNTIME_B"], "b_runtime_function_BLACK.xlsx"),
    "RUN_COMPARE_B"   : os.path.join(DIRS["RUNTIME_B"], "c_runtime_compare_BLACK.xlsx"),

    # Submission
    "SUBMISSION"      : os.path.join(DIRS["LOGS"], "06_submission_report.xlsx"),

    # AST & Graph
    "EXTRACT_AST_P"     : os.path.join(DIRS["LOGS"], "07_AST_report_prak.xlsx"),
    "EXTRACT_AST_T"     : os.path.join(DIRS["LOGS"], "07_AST_report_tgs.xlsx"),
    "CONSTRUCT_GRAPH_P" : os.path.join(DIRS["LOGS"], "08_Graph_report_prak.xlsx"),
    "CONSTRUCT_GRAPH_T" : os.path.join(DIRS["LOGS"], "08_Graph_report_tgs.xlsx"),
    "LIST_GRAPH_P"      : os.path.join(DIRS["LOGS"], "09_list_Graph_report_prak.xlsx"),
    "LIST_GRAPH_T"      : os.path.join(DIRS["LOGS"], "09_list_Graph_report_tgs.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT_P": os.path.join(DIRS["LOGS"], "10_embedding_report_prak.xlsx"),
    "EMBEDDING_REPORT_T": os.path.join(DIRS["LOGS"], "10_embedding_report_tgs.xlsx"),
    "EMBEDDING_VECTOR_P": os.path.join(DIRS["LOGS"], "10a_embedding_vector_prak.xlsx"),
    "EMBEDDING_VECTOR_T": os.path.join(DIRS["LOGS"], "10a_embedding_vector_tgs.xlsx"),

    # Similarity PRAKTIKUM
    "SIMILARITY_P"      : os.path.join(DIRS["LOGS"], "11_similarity_report_prak.xlsx"),
    "SIMILARITY_MODUL_P" : os.path.join(DIRS['LOGS'], "11a_similarity_per_Modul_prak.xlsx"),
    "SIMILARITY_SUMMARY_P" : os.path.join(DIRS['LOGS'], "11b_similarity_summary_prak.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN_P" : os.path.join(DIRS['LOGS'], "12_euclidean_similarity_prak.xlsx"),
    "EUCLIDEAN_MODUL_P" : os.path.join(DIRS['LOGS'], "12a_euclidean_similarity_modul_prak.xlsx"),
    
    "SIMILARITY_COMPARE_P" : os.path.join(DIRS['LOGS'], "13_similarity_comparison_prak.xlsx"),
    
    # Similarity TUGAS
    "SIMILARITY_T"      : os.path.join(DIRS["LOGS"], "11_similarity_report_tgs.xlsx"),
    "SIMILARITY_MODUL_T" : os.path.join(DIRS['LOGS'], "11a_similarity_per_Modul_tgs.xlsx"),
    "SIMILARITY_SUMMARY_T" : os.path.join(DIRS['LOGS'], "11b_similarity_summary_tgs.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN_T" : os.path.join(DIRS['LOGS'], "12_euclidean_similarity_tgs.xlsx"),
    "EUCLIDEAN_MODUL_T" : os.path.join(DIRS['LOGS'], "12a_euclidean_similarity_modul_tgs.xlsx"),
    
    "SIMILARITY_COMPARE_T" : os.path.join(DIRS['LOGS'], "13_similarity_comparison_tgs.xlsx"),
    
    # METRICS
    "METRICS"         : os.path.join(DIRS["LOGS"], "metrics_evaluation_report.xlsx"),
    "EXECUTION_TIME"  : os.path.join(DIRS["LOGS"], "execution_time_report.xlsx"),
}

# VALIDATION & DIRECTORY INITIALIZATION
print("=" * 70)
print(f"{'DIRECTORY CONFIGURATION':^70}")
print("=" * 70)

# Validasi root directory
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"❌ BASE_DIR tidak ditemukan: {BASE_DIR}")

# Validasi file input
if not os.path.isfile(INPUT_GITHUB):
    raise FileNotFoundError(f"❌ File input tidak ditemukan: {INPUT_GITHUB}")

if not INPUT_GITHUB.endswith(".txt"):
    raise ValueError("❌ File input harus berekstensi .txt")

if os.path.getsize(INPUT_GITHUB) == 0:
    raise ValueError(f"❌ File input kosong: {INPUT_GITHUB}")

# Membuat folder pipeline
folders_created = 0

for name, path in DIRS.items():
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        folders_created += 1
        status = "[NEW]"
    elif not os.path.isdir(path):
        raise NotADirectoryError(f"❌ Path bukan folder: {path}")
    else:
        status = "[EXISTS]"
    print(f"{status:<10} {name:<12} : {path}")

# Validasi parent folder output
missing_results_parent = []

for name, path in RESULTS.items():
    parent_dir = os.path.dirname(path)
    if not os.path.exists(parent_dir):
        missing_results_parent.append(
            f"{name} : {parent_dir}"
        )

if missing_results_parent:
    raise FileNotFoundError(
        "❌ Parent folder RESULTS tidak ditemukan:\n"
        + "\n".join(missing_results_parent)
    )

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("-" * 70)
print(f"Folders Created  : {folders_created}")
print(f"Dataset Root     : {DIRS['DATASET']}")
print(f"Input File       : {INPUT_GITHUB}")
print("=" * 70)
print(f"Diproses pada {timestamp}")

                       DIRECTORY CONFIGURATION                        
[EXISTS]   DATASET      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)
[EXISTS]   RAW          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\00_Raw
[EXISTS]   ANON         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\01_Raw_Anon
[EXISTS]   NORM         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\02_Normalized
[EXISTS]   CONV         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\03_Converted
[EXISTS]   CLEAN        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
[EXISTS]   AUTOPEP8     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05a_Autopep8
[EXISTS]   BLACK        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05b_Black
[EXISTS]   FILTERED     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered
[EXISTS]   FILTERED_PRAK : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_prak
[EXISTS]   FILTERED_TGS : D:\PUTRI\D4\S

In [3]:
# ==============================================================================
# EVALUATION CONFIG DIR
# ==============================================================================
DATASET_EVAL_FOLDER = DATASET_FOLDER + "_eval"
DATASET_EVAL_DIR = os.path.join(BASE_DIR, DATASET_EVAL_FOLDER)
DATASET_EVAL_P = os.path.join(DATASET_EVAL_DIR, "praktikum")
DATASET_EVAL_T = os.path.join(DATASET_EVAL_DIR, "tugas")

EVAL_P = {
    # PRAKTIKUM
    "SAMPLE"       : os.path.join(DATASET_EVAL_P, "06_SAMPLE"),
    "AST"          : os.path.join(DATASET_EVAL_P, "07_AST"),
    "AST_VISUAL"   : os.path.join(DATASET_EVAL_P, "07a_AST_visual"),
    "GRAPH"        : os.path.join(DATASET_EVAL_P, "08_Graph"),
    "INPUT_GRAPH"  : os.path.join(DATASET_EVAL_P, "09_Graph2vec_Input"),
    "EMBEDDING"    : os.path.join(DATASET_EVAL_P, "10_Graph2vec_Embedding"),
}

EVAL_T = {
    # TUGAS
    "SAMPLE"       : os.path.join(DATASET_EVAL_T, "06_SAMPLE"),
    "AST"          : os.path.join(DATASET_EVAL_T, "07_AST"),
    "AST_VISUAL"   : os.path.join(DATASET_EVAL_T, "07a_AST_visual"),
    "GRAPH"        : os.path.join(DATASET_EVAL_T, "08_Graph"),
    "INPUT_GRAPH"  : os.path.join(DATASET_EVAL_T, "09_Graph2vec_Input"),
    "EMBEDDING"    : os.path.join(DATASET_EVAL_T, "10_Graph2vec_Embedding"),
}

RESULTS_EVAL_P = {
    "SAMPLING_REPORT" : os.path.join(DIRS["EVAL_P"], "06_sampling_report.xlsx"),
    # AST & Graph
    "EXTRACT_AST"     : os.path.join(DIRS["EVAL_P"], "07_AST_report.xlsx"),
    "CONSTRUCT_GRAPH" : os.path.join(DIRS["EVAL_P"], "08_Graph_report.xlsx"),
    "LIST_GRAPH"      : os.path.join(DIRS["EVAL_P"], "09_list_Graph_report.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT": os.path.join(DIRS["EVAL_P"], "10_embedding_report.xlsx"),
    "EMBEDDING_VECTOR": os.path.join(DIRS["EVAL_P"], "10a_embedding_vector.xlsx"),

    # Similarity
    "SIMILARITY"      : os.path.join(DIRS["EVAL_P"], "11_similarity_report.xlsx"),
    "SIMILARITY_MODUL" : os.path.join(DIRS['EVAL_P'], "11a_similarity_per_Modul.xlsx"),
    "SIMILARITY_SUMMARY" : os.path.join(DIRS['EVAL_P'], "11b_similarity_summary.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN" : os.path.join(DIRS['EVAL_P'], "12_euclidean_similarity.xlsx"),
    "EUCLIDEAN_MODUL" : os.path.join(DIRS['EVAL_P'], "12a_euclidean_similarity_modul.xlsx"),
    
    "SIMILARITY_COMPARE" : os.path.join(DIRS['EVAL_P'], "13_similarity_comparison.xlsx"),
}

RESULTS_EVAL_T = {
    "SAMPLING_REPORT" : os.path.join(DIRS["EVAL_T"], "06_sampling_report.xlsx"),
    # AST & Graph
    "EXTRACT_AST"     : os.path.join(DIRS["EVAL_T"], "07_AST_report.xlsx"),
    "CONSTRUCT_GRAPH" : os.path.join(DIRS["EVAL_T"], "08_Graph_report.xlsx"),
    "LIST_GRAPH"      : os.path.join(DIRS["EVAL_T"], "09_list_Graph_report.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT": os.path.join(DIRS["EVAL_T"], "10_embedding_report.xlsx"),
    "EMBEDDING_VECTOR": os.path.join(DIRS["EVAL_T"], "10a_embedding_vector.xlsx"),

    # Similarity
    "SIMILARITY"      : os.path.join(DIRS["EVAL_T"], "11_similarity_report.xlsx"),
    "SIMILARITY_MODUL" : os.path.join(DIRS['EVAL_T'], "11a_similarity_per_Modul.xlsx"),
    "SIMILARITY_SUMMARY" : os.path.join(DIRS['EVAL_T'], "11b_similarity_summary.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN" : os.path.join(DIRS['EVAL_T'], "12_euclidean_similarity.xlsx"),
    "EUCLIDEAN_MODUL" : os.path.join(DIRS['EVAL_T'], "12a_euclidean_similarity_modul.xlsx"),
    
    "SIMILARITY_COMPARE" : os.path.join(DIRS['EVAL_T'], "13_similarity_comparison.xlsx"),
}


# INITIALIZATION & VALIDATION EVALUATION DIRECTORY
folders_created = 0
print(f"{'='*20} EVALUATION DIRECTORY CONFIGURATION {'='*20}")

# Validasi dan pembuatan folder evaluasi
for cfg_name, config in [("EVAL_P", EVAL_P), ("EVAL_T", EVAL_T)]:
    print(f"\n[{cfg_name}]")
    for name, path in config.items():
        if not os.path.exists(path):
            os.makedirs(path, exist_ok=True)
            folders_created += 1
            status = "[NEW]"
        elif not os.path.isdir(path):
            raise NotADirectoryError(f"❌ Path bukan folder: {path}")
        else:
            status = "[EXISTS]"
        print(f"{status:<10} {name:<15} : {path}")

# ------------------------------------------------------------------------------
# VALIDASI PARENT DIRECTORY UNTUK FILE REPORT
# ------------------------------------------------------------------------------
missing_results = []
for grp_name, group in [("RESULTS_EVAL_P", RESULTS_EVAL_P), ("RESULTS_EVAL_T", RESULTS_EVAL_T)]:
    for name, path in group.items():
        parent_dir = os.path.dirname(path)
        if not os.path.exists(parent_dir):
            missing_results.append(f"{grp_name}.{name} -> {parent_dir}")

if missing_results:
    raise FileNotFoundError("❌ Parent folder RESULTS tidak ditemukan:\n" + "\n".join(missing_results))

# ------------------------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------------------------
print(f"\n{'-'*70}")
print(f"Folders Created     : {folders_created}")
print(f"Evaluation Root     : {DATASET_EVAL_DIR}")
print(f"Praktikum Root      : {DATASET_EVAL_P}")
print(f"Tugas Root          : {DATASET_EVAL_T}")
print("=" * 70)
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

==================== EVALUATION DIRECTORY CONFIGURATION ====================

[EVAL_P]
[EXISTS]   SAMPLE          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE
[EXISTS]   AST             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07_AST
[EXISTS]   AST_VISUAL      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07a_AST_visual
[EXISTS]   GRAPH           : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\08_Graph
[EXISTS]   INPUT_GRAPH     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\09_Graph2vec_Input
[EXISTS]   EMBEDDING       : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\10_Graph2vec_Embedding

[EVAL_T]
[EXISTS]   SAMPLE          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE
[EXISTS]   AST             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\07_AST
[EXISTS]   AST_VISUAL      : D:\PUTRI\D4\SE

## Helper Function

### helper count

In [4]:
# HELPER FUNCTION count

def count_all_files(dataset_path, extensions=None, exclude_dirs=None):
    total_files = 0
    by_extension = {}

    exclude_dirs = set(exclude_dirs or [])

    for root, dirs, files in os.walk(dataset_path):
        # skip folder tertentu
        dirs[:] = [d for d in dirs if d not in exclude_dirs]

        for file in files:
            ext = os.path.splitext(file)[1].lower()

            # filter ekstensi jika diberikan
            if extensions and ext not in extensions:
                continue

            total_files += 1
            by_extension[ext] = by_extension.get(ext, 0) + 1

    return {
        "total_files": total_files,
        "by_extension": by_extension
    }

import os
import pandas as pd

def count_students(data, nim_column="nim"):
    """
    Menghitung jumlah mahasiswa dari berbagai tipe data.
    """

    # Folder dataset
    if isinstance(data, str):
        return sum(
            1
            for item in os.listdir(data)
            if os.path.isdir(os.path.join(data, item))
        )

    # DataFrame
    if isinstance(data, pd.DataFrame):
        return data[nim_column].nunique()

    # List metadata
    if isinstance(data, list):
        return len({
            row[nim_column]
            for row in data
            if nim_column in row
        })

    raise TypeError(
        "data harus berupa path folder, DataFrame, atau list metadata"
    )
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-06-11 22:45:08


In [5]:
def extract_assignment_name(filename):
    """
    NIM_MODUL_FILE.py
    -> FILE
    """
    filename = os.path.splitext(
        os.path.basename(filename)
    )[0]
    parts = filename.split("_")
    if len(parts) < 3:
        return None, None, filename
    nim = parts[0]
    modul = parts[1]
    nama_file = "_".join(parts[2:])
    return nim, modul, nama_file

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-06-11 22:45:08


In [6]:
def get_path_size(path, unit="kb"):
    """Menghitung kapasitas memori file/folder secara rekursif."""
    if not os.path.exists(path):
        return 0

    total_size = 0
    if os.path.isfile(path):
        try: total_size = os.path.getsize(path)
        except Exception: return 0
    else:
        for root, _, files in os.walk(path):
            for file in files:
                file_path = os.path.join(root, file)
                try:
                    if os.path.exists(file_path): total_size += os.path.getsize(file_path)
                except Exception: pass

    unit = unit.lower()
    factors = {"byte": 1, "kb": 1024, "mb": 1024**2, "gb": 1024**3}
    if unit in factors:
        return total_size if unit == "byte" else round(total_size / factors[unit], 2)
    else:
        raise ValueError("Unit tidak valid. Gunakan: byte/kb/mb/gb")


def count_loc(file_path):
    """Menghitung kuantitas baris kode aktif (tanpa baris kosong)."""
    if not os.path.exists(file_path):
        return 0

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".py":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                return sum(1 for line in f if line.strip())
        elif ext == ".ipynb":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                notebook = json.load(f)
            loc = 0
            for cell in notebook.get("cells", []):
                if cell.get("cell_type") == "code":
                    source = cell.get("source", [])
                    loc += sum(1 for line in source if str(line).strip())
            return loc
        return 0
    except Exception:
        return 0

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-06-11 22:45:08


### helper timer

In [7]:
# ==============================================================================
# RUNTIME METRICS
# ==============================================================================
from openpyxl.styles import Alignment, Font

def start_timer():
    """Memulai timer komputasi."""
    return time.perf_counter()

def save_execution_time(start_time, stage, total_mahasiswa, total_file, total_size_kb, output_file=RESULTS["EXECUTION_TIME"]):
    """
    Simpan atau update metrik runtime berdasarkan stage ke Excel.
    """
    execution_time = round(time.perf_counter() - start_time, 2)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    new_data = {
        "stage": stage,
        "total_mahasiswa": total_mahasiswa,
        "total_file": total_file,
        "waktu_eksekusi_s": execution_time,
        "total_size_kb": round(total_size_kb, 2),
        "timestamp": timestamp
    }

    # Logika Update/Insert
    if os.path.exists(output_file):
        df = pd.read_excel(output_file)
        if not df.empty and stage in df["stage"].values:
            df.loc[
                df["stage"] == stage,
                [
                    "total_mahasiswa",
                    "total_file",
                    "waktu_eksekusi_s",
                    "total_size_kb",
                    "timestamp"
                ]
            ] = [
                total_mahasiswa,
                total_file,
                execution_time,
                round(total_size_kb, 2),
                timestamp
            ]
        else:
            df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)
    else:
        df = pd.DataFrame([new_data])

    df = df.sort_values(by="stage").reset_index(drop=True)
    
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Execution Time")

        ws = writer.sheets["Execution Time"]

        ws.freeze_panes = "A2"
        # Header bold + center
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )
        for col in ws.columns:
            max_len = max(
                (
                    len(str(cell.value))
                    if cell.value is not None
                    else 0
                )
                for cell in col
            )
            ws.column_dimensions[
                col[0].column_letter
            ].width = min(max_len + 4, 60)

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="center")
            
    print(f"\n[SELESAI] {stage} | {execution_time} s | {total_file} file")
    return execution_time

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Runtime Metrics Logger siap digunakan pada {timestamp}")

✅ Runtime Metrics Logger siap digunakan pada 2026-06-11 22:45:08


# AST (Abstract Syntax Tree)

## 7. Ekstraksi AST

###     Ekstraks AST (py -> JSON)

In [8]:
# FUNCTION HELPER - AST EXTRACTION
# ==============================================================================

# Filter file berdasarkan requirement modul
def filter_files_by_requirement(py_files, input_dir, req_dict):
    filtered = []
    for file_path in py_files:
        rel_path = os.path.relpath(file_path, input_dir)
        parts = rel_path.split(os.sep)

        if len(parts) < 3:
            continue

        nim, modul, filename = parts[0], parts[1], parts[-1]
        if modul in req_dict and filename in req_dict[modul]:
            filtered.append(file_path)

    return filtered

# Function untuk logging
def save_log(nim, modul, file, metrics=None, status="SUCCESS", message=""):
    m = metrics if metrics else {
        "depth": 0, "node_count": 0, "function_count": 0, 
        "variable_count": 0, "cyclomatic_complexity": 0, "loc": 0
    }

    return {
        "nim": nim,
        "modul": modul,
        "file": file,
        "kedalaman_ast": m["depth"],
        "jumlah_node": m["node_count"],
        "jumlah_function": m["function_count"],
        "jumlah_variabel_unik": m["variable_count"],
        "cyclomatic_complexity": m["cyclomatic_complexity"],
        "loc": m["loc"],
        "status": status,
        "keterangan": message
    }
    
# READ FILE
# membaca isi file .py
# ------------------------------------------------------------------------------
def read_python_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read(), None
    except Exception as e:
        return None, str(e)

# PARSE AST 
# mengubah kode ke bentuk AST (objek), dan cek jika syntax error
# ------------------------------------------------------------------------------
def parse_ast(code):
    try:
        tree = ast.parse(code)
        return tree, None, None
    except SyntaxError as e:
        return None, "SYNTAX_ERROR", f"SyntaxError: {str(e)}"
    except Exception as e:
        return None, "FAILED", str(e)

# AST TO DICT (RECURSIVE)
# mengubah objek AST ke dict untuk nantinya disimpan ke JSON
# ------------------------------------------------------------------------------
def ast_to_dict(node):
    if isinstance(node, ast.AST):
        result = {"type": type(node).__name__}
        if hasattr(node, "lineno"):
            result["lineno"] = node.lineno
        if hasattr(node, "end_lineno"):
            result["end_lineno"] = node.end_lineno
        for field, value in ast.iter_fields(node):
            result[field] = ast_to_dict(value)
        return result
    elif isinstance(node, list):
        return [ast_to_dict(item) for item in node]
    else:
        # HANDLE NON-SERIALIZABLE TYPES
        if node is Ellipsis:
            return "Ellipsis"

        if isinstance(node, (str, int, float, bool)) or node is None:
            return node
        
        return str(node)

# METRICS EXTRACTION
# menghitung kedalaman AST, jumlah node, jumlah function, jumlah variabel
# ------------------------------------------------------------------------------
def compute_ast_metrics(tree, code):
    max_depth = 0
    node_count = 0
    function_count = 0
    cc_score = 1
    unique_vars = set() # Menggunakan set untuk menyimpan nama unik

    # Daftar node yang menambah Cyclomatic Complexity
    # (if, while, for, except, with, logical operators)
    cc_nodes = (
        ast.If, ast.While, ast.For, ast.AsyncFor, 
        ast.ExceptHandler, ast.With, ast.AsyncWith,
        ast.IfExp, ast.Assert
    )
    
    def visit(node, depth=0):
        nonlocal max_depth, node_count, function_count, cc_score

        if isinstance(node, ast.AST):
            node_count += 1
            max_depth = max(max_depth, depth)
            
            # Hitung Cyclomatic Complexity
            if isinstance(node, cc_nodes):
                cc_score += 1
            # Tambahan untuk BoolOp (and/or bisa punya banyak values)
            if isinstance(node, ast.BoolOp):
                cc_score += len(node.values) - 1

            # hitung function
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda)): 
                function_count += 1

            # hitung variabel unik
            if isinstance(node, ast.Name):
                if isinstance(node.ctx, (ast.Store, ast.Param)):
                    unique_vars.add(node.id)

            for child in ast.iter_child_nodes(node):
                visit(child, depth + 1)

    visit(tree)
    
    # hitung loc
    # loc = len(code.split('\n'))
    loc = sum(1 for line in code.split('\n') if line.strip())
    
    return {
        "depth": max_depth,
        "node_count": node_count,
        "function_count": function_count,
        "variable_count": len(unique_vars), # Jumlah elemen unik di set
        "cyclomatic_complexity": cc_score,
        "loc": loc
    }

# SAVE JSON
# menyimpan hasil ekstraksi AST ke JSON
# ------------------------------------------------------------------------------
def save_json(data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# EXTRACT METADATA (nim, modul, filename)
# ------------------------------------------------------------------------------
def extract_metadata(root_dir, filepath):
    relative_path = os.path.relpath(filepath, root_dir)
    parts = relative_path.split(os.sep)
    filename = os.path.splitext(os.path.basename(filepath))[0]
    
    # FORMAT DATASET UTAMA    NIM/JS01/file.py
    if len(parts) >= 3:
        nim = parts[0]
        modul = parts[1]
    # FORMAT EVALUASI type1/22001_JS01_file_type1.py
    elif len(parts) == 2:
        file_parts = filename.split("_")
        nim = file_parts[0] if len(file_parts) >= 1 else "unknown"
        modul = file_parts[1] if len(file_parts) >= 2 else "unknown"
    else:
        nim = "unknown"
        modul = "unknown"
    return nim, modul, filename

# collect files python
def collect_py_files(input_dir):
    py_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".py"):
                py_files.append(os.path.join(root, file))
    return py_files

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Helper functions dijalankan pada {timestamp}")

✅ Helper functions dijalankan pada 2026-06-11 22:45:31


In [9]:
# RUN & EXECUTION
# ==============================================================================

def process_file(file_path, input_dir, output_dir):
    nim, modul, filename = extract_metadata(input_dir, file_path)
    mhs_id, modul_id, file_id = extract_assignment_name(filename)
    code, read_err = read_python_file(file_path)
    if read_err:
        return save_log(nim, modul, filename, status="FAILED", message=read_err)

    tree, err_type, msg = parse_ast(code)

    if err_type:
        return save_log(nim, modul, filename, status=err_type, message=msg)

    try:
        metrics = compute_ast_metrics(tree, code)
        # AST → JSON
        ast_dict = ast_to_dict(tree)
        json_content = {
            "nim": nim,
            "modul": modul,
            "filename": file_id,
            "metrics": {
                "kedalaman_ast": metrics["depth"],
                "jumlah_node": metrics["node_count"],
                "jumlah_function": metrics["function_count"],
                "jumlah_variabel_unik": metrics["variable_count"],
                "cyclomatic_complexity": metrics["cyclomatic_complexity"],
                "loc": metrics["loc"]
            },
            "ast": ast_dict  # Struktur AST Utama
        }
        relative_path = os.path.relpath(file_path, input_dir)
        output_path = os.path.join(output_dir, relative_path.replace(".py", ".json"))
        save_json(json_content, output_path)
        
        json_size = get_path_size(output_path, unit='kb')
        file_loc = count_loc(file_path)
        return save_log(nim, modul, filename, metrics=metrics)
    except Exception as e:
        return save_log(nim, modul, filename, status="FAILED", message=str(e))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Helper functions dijalankan pada {timestamp}")

✅ Helper functions dijalankan pada 2026-06-11 22:45:31


In [10]:
# EXECUTION
# --------------------------
print("="*60)
print(f"{' EXTRACT AST PRAKTIKUM':^60}")
print("-"*60)

INPUT_DIR = DIRS['FILTERED_PRAK']
OUTPUT_DIR = DIRS['AST_P']
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"{'Dataset input':<30}: {INPUT_DIR}")

# STEP 1: scan untuk dapat requirement
# df_raw = scan_directory(INPUT_DIR)
# df_raw = df_raw[df_raw['modul'].str.lower() != 'unclassified']

# req_dict = extract_requirements_from_dataset(df_raw)

# STEP 2: ambil semua file
all_py_files = collect_py_files(INPUT_DIR)

# STEP 3: filter hanya requirement
# filtered_py_files = filter_files_by_requirement(all_py_files, INPUT_DIR, req_dict)

print(f"{'Total File input':<30}: {len(all_py_files)}")
# print(f"{'Total File setelah filter':<30}: {len(filtered_py_files)}")

logs = []
success = 0
failed = 0
syntax_err = 0

extract_start = start_timer()
total_loc = 0

for file_path in tqdm(all_py_files, desc="Extracting AST", unit="file"):
    log = process_file(file_path, INPUT_DIR, OUTPUT_DIR)
    logs.append(log)

    # update counter
    if log["status"] == "SUCCESS":
        success += 1
        if "loc" in log:
            total_loc += log["loc"]
    elif log["status"] == "FAILED":
        failed += 1
    elif log["status"] == "SYNTAX_ERROR":
        syntax_err += 1

# 1. Konversi logs ke DataFrame
df_logs = pd.DataFrame(logs)

print(f"{'AST disimpan di':<30}: {OUTPUT_DIR}")
total_mhs = count_students(df_logs)
total_files = count_all_files(OUTPUT_DIR)['total_files']
extract_time = save_execution_time(extract_start, "7_Ekstraksi_AST_Praktikum", total_mahasiswa=total_mhs, total_file=total_files, total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# 2. Filter data, ambil yang statusnya sukses
df_success = df_logs[df_logs['status'] == 'SUCCESS']
# 3. Simpan Report ke Excel
report_path = RESULTS['EXTRACT_AST_P']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

# 4. Perhitungan Ringkasan Dasar (Logic Corrected)
total_input = len(all_py_files) # Total file .py di folder input
total_processed = len(df_logs)  # Total file yang masuk ke loop processing

# Success Rate dihitung dari (SUCCESS / TOTAL INPUT)
success_rate = (success / total_input * 100) if total_input > 0 else 0

display(HTML(f"<h3>RINGKASAN HASIL EKSTRAKSI AST</h3>"))
print("=" * 75)
print(f"{'Dataset Input':<35}: {INPUT_DIR}")
print(f"{'Total Mahasiswa':<35}: {total_mhs}")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total File Diproses':<35}: {total_processed}")
print(f"{'Waktu Eksekusi':<35}: {extract_time} seconds")
print(f"{'Total File JSON disimpan':<35}: {count_all_files(OUTPUT_DIR, exclude_dirs=['eval'])['total_files']}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal (FAILED)':<35}: {failed}")
print(f"{'Dilewati (SYNTAX ERROR)':<35}: {syntax_err}")
print(f"{'Success Rate':<35}: {success_rate:.2f}%")

print("\n" + "-" * 75)
print(f"Report Excel  : {report_path}")
print(f"Dataset JSON  : {OUTPUT_DIR}")

# Menampilkan 10 sampel teratas yang sukses
if not df_success.empty:
    print("\nSamples Data Sukses:")
    display(df_success.head(10))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                    EXTRACT AST PRAKTIKUM                   
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_prak
Total File input              : 1678


Extracting AST:   0%|          | 0/1678 [00:00<?, ?file/s]

AST disimpan di               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST_prak

[SELESAI] 7_Ekstraksi_AST_Praktikum | 91.95 s | 1678 file


Dataset Input                      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_prak
Total Mahasiswa                    : 53
Total File Input                   : 1678
Total File Diproses                : 1678
Waktu Eksekusi                     : 91.95 seconds
Total File JSON disimpan           : 1678
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 1678
Gagal (FAILED)                     : 0
Dilewati (SYNTAX ERROR)            : 0
Success Rate                       : 100.00%

---------------------------------------------------------------------------
Report Excel  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\07_AST_report_prak.xlsx
Dataset JSON  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST_prak

Samples Data Sukses:


,nim,modul,file,kedalaman_ast,jumlah_node,jumlah_function,jumlah_variabel_unik,cyclomatic_complexity,loc,status,keterangan
0,MHS001,js02,MHS001_js02_p01,9,415,0,9,3,47,SUCCESS,
1,MHS001,js02,MHS001_js02_p02,9,113,0,1,1,12,SUCCESS,
2,MHS001,js02,MHS001_js02_p03,7,212,0,11,1,29,SUCCESS,
3,MHS001,js02,MHS001_js02_p04,5,142,0,4,1,18,SUCCESS,
4,MHS001,js03,MHS001_js03_p01,13,501,0,3,1,86,SUCCESS,
5,MHS001,js03,MHS001_js03_p02,5,142,0,4,1,18,SUCCESS,
6,MHS001,js03,MHS001_js03_p03,7,224,0,4,1,22,SUCCESS,
7,MHS001,js03,MHS001_js03_p04,7,53,0,4,1,6,SUCCESS,
8,MHS001,js04,MHS001_js04_p01,7,360,0,12,3,38,SUCCESS,
9,MHS001,js04,MHS001_js04_p02,11,1583,2,42,11,153,SUCCESS,


Diproses pada 2026-06-11 22:47:06


In [11]:
# EXECUTION TUGAS
# --------------------------
print("="*60)
print(f"{' EXTRACT AST TUGAS':^60}")
print("-"*60)

INPUT_DIR = DIRS['FILTERED_TGS']
OUTPUT_DIR = DIRS['AST_T']
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"{'Dataset input':<30}: {INPUT_DIR}")

all_py_files = collect_py_files(INPUT_DIR)

print(f"{'Total File input':<30}: {len(all_py_files)}")

logs = []
success = 0
failed = 0
syntax_err = 0

extract_start = start_timer()
total_loc = 0

for file_path in tqdm(all_py_files, desc="Extracting AST", unit="file"):
    log = process_file(file_path, INPUT_DIR, OUTPUT_DIR)
    logs.append(log)

    # update counter
    if log["status"] == "SUCCESS":
        success += 1
        if "loc" in log:
            total_loc += log["loc"]
    elif log["status"] == "FAILED":
        failed += 1
    elif log["status"] == "SYNTAX_ERROR":
        syntax_err += 1

# 1. Konversi logs ke DataFrame
df_logs = pd.DataFrame(logs)

print(f"{'AST disimpan di':<30}: {OUTPUT_DIR}")
total_mhs = count_students(df_logs)
total_files = count_all_files(OUTPUT_DIR)['total_files']
extract_time = save_execution_time(extract_start, "7_Ekstraksi_AST_Tugas", total_mahasiswa=total_mhs, total_file=total_files, total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# 2. Filter data, ambil yang statusnya sukses
df_success = df_logs[df_logs['status'] == 'SUCCESS']
# 3. Simpan Report ke Excel
report_path = RESULTS['EXTRACT_AST_T']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

# 4. Perhitungan Ringkasan Dasar (Logic Corrected)
total_input = len(all_py_files) # Total file .py di folder input
total_processed = len(df_logs)  # Total file yang masuk ke loop processing

# Success Rate dihitung dari (SUCCESS / TOTAL INPUT)
success_rate = (success / total_input * 100) if total_input > 0 else 0

display(HTML(f"<h3>RINGKASAN HASIL EKSTRAKSI AST</h3>"))
print("=" * 75)
print(f"{'Dataset Input':<35}: {INPUT_DIR}")
print(f"{'Total Mahasiswa':<35}: {total_mhs}")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total File Diproses':<35}: {total_processed}")
print(f"{'Waktu Eksekusi':<35}: {extract_time} seconds")
print(f"{'Total File JSON disimpan':<35}: {count_all_files(OUTPUT_DIR, exclude_dirs=['eval'])['total_files']}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal (FAILED)':<35}: {failed}")
print(f"{'Dilewati (SYNTAX ERROR)':<35}: {syntax_err}")
print(f"{'Success Rate':<35}: {success_rate:.2f}%")

print("\n" + "-" * 75)
print(f"Report Excel  : {report_path}")
print(f"Dataset JSON  : {OUTPUT_DIR}")

# Menampilkan 10 sampel teratas yang sukses
if not df_success.empty:
    print("\nSamples Data Sukses:")
    display(df_success.head(10))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                      EXTRACT AST TUGAS                     
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_tgs
Total File input              : 566


Extracting AST:   0%|          | 0/566 [00:00<?, ?file/s]

AST disimpan di               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST_tgs

[SELESAI] 7_Ekstraksi_AST_Tugas | 28.53 s | 566 file


Dataset Input                      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_tgs
Total Mahasiswa                    : 50
Total File Input                   : 566
Total File Diproses                : 566
Waktu Eksekusi                     : 28.53 seconds
Total File JSON disimpan           : 566
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 566
Gagal (FAILED)                     : 0
Dilewati (SYNTAX ERROR)            : 0
Success Rate                       : 100.00%

---------------------------------------------------------------------------
Report Excel  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\07_AST_report_tgs.xlsx
Dataset JSON  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST_tgs

Samples Data Sukses:


,nim,modul,file,kedalaman_ast,jumlah_node,jumlah_function,jumlah_variabel_unik,cyclomatic_complexity,loc,status,keterangan
0,MHS001,js02,MHS001_js02_tp,11,655,0,12,7,67,SUCCESS,
1,MHS001,js03,MHS001_js03_tp,11,1216,0,19,9,124,SUCCESS,
2,MHS001,js04,MHS001_js04_tp,10,2815,0,64,25,384,SUCCESS,
3,MHS001,js13,MHS001_js13_tp,8,2094,0,45,4,274,SUCCESS,
4,MHS001,js14,MHS001_js14_tp,8,1952,0,28,7,250,SUCCESS,
5,MHS002,js02,MHS002_js02_tp,9,248,0,6,3,32,SUCCESS,
6,MHS002,js03,MHS002_js03_tp,6,215,0,11,1,26,SUCCESS,
7,MHS002,js04,MHS002_js04_tp,9,1128,0,37,11,127,SUCCESS,
8,MHS002,js05,MHS002_js05_tp,6,318,0,12,2,39,SUCCESS,
9,MHS002,js06,MHS002_js06_tp,9,541,0,21,2,74,SUCCESS,


Diproses pada 2026-06-11 22:47:36


In [12]:
# EXECUTION PRAKTIKUM EVAL
# --------------------------
print("="*60)
print(f"{' EXTRACT AST EVALUASI PRAKTIKUM':^60}")
print("-"*60)

INPUT_DIR = EVAL_P['SAMPLE']
OUTPUT_DIR = EVAL_P['AST']
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"{'Dataset input':<30}: {INPUT_DIR}")

all_py_files = collect_py_files(INPUT_DIR)

print(f"{'Total File input':<30}: {len(all_py_files)}")

logs = []
success = 0
failed = 0
syntax_err = 0

extract_start = start_timer()
total_loc = 0

for file_path in tqdm(all_py_files, desc="Extracting AST", unit="file"):
    log = process_file(file_path, INPUT_DIR, OUTPUT_DIR)
    logs.append(log)

    # update counter
    if log["status"] == "SUCCESS":
        success += 1
        if "loc" in log:
            total_loc += log["loc"]
    elif log["status"] == "FAILED":
        failed += 1
    elif log["status"] == "SYNTAX_ERROR":
        syntax_err += 1

# 1. Konversi logs ke DataFrame
df_logs = pd.DataFrame(logs)

print(f"{'AST disimpan di':<30}: {OUTPUT_DIR}")
total_mhs = count_students(df_logs)
total_files = count_all_files(OUTPUT_DIR)['total_files']
extract_time = save_execution_time(extract_start, "7_Ekstraksi_AST_EVAL_Praktikum", total_mahasiswa=total_mhs, total_file=total_files, total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# 2. Filter data, ambil yang statusnya sukses
df_success = df_logs[df_logs['status'] == 'SUCCESS']
# 3. Simpan Report ke Excel
report_path = RESULTS_EVAL_P['EXTRACT_AST']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

# 4. Perhitungan Ringkasan Dasar (Logic Corrected)
total_input = len(all_py_files) # Total file .py di folder input
total_processed = len(df_logs)  # Total file yang masuk ke loop processing

# Success Rate dihitung dari (SUCCESS / TOTAL INPUT)
success_rate = (success / total_input * 100) if total_input > 0 else 0

display(HTML(f"<h3>RINGKASAN HASIL EKSTRAKSI AST</h3>"))
print("=" * 75)
print(f"{'Dataset Input':<35}: {INPUT_DIR}")
print(f"{'Total Mahasiswa':<35}: {total_mhs}")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total File Diproses':<35}: {total_processed}")
print(f"{'Waktu Eksekusi':<35}: {extract_time} seconds")
print(f"{'Total File JSON disimpan':<35}: {count_all_files(OUTPUT_DIR, exclude_dirs=['eval'])['total_files']}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal (FAILED)':<35}: {failed}")
print(f"{'Dilewati (SYNTAX ERROR)':<35}: {syntax_err}")
print(f"{'Success Rate':<35}: {success_rate:.2f}%")

print("\n" + "-" * 75)
print(f"Report Excel  : {report_path}")
print(f"Dataset JSON  : {OUTPUT_DIR}")

# Menampilkan 10 sampel teratas yang sukses
if not df_success.empty:
    print("\nSamples Data Sukses:")
    display(df_success.head(10))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

               EXTRACT AST EVALUASI PRAKTIKUM               
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE
Total File input              : 2515


Extracting AST:   0%|          | 0/2515 [00:00<?, ?file/s]

AST disimpan di               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07_AST

[SELESAI] 7_Ekstraksi_AST_EVAL_Praktikum | 100.32 s | 2515 file


Dataset Input                      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE
Total Mahasiswa                    : 52
Total File Input                   : 2515
Total File Diproses                : 2515
Waktu Eksekusi                     : 100.32 seconds
Total File JSON disimpan           : 2515
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 2515
Gagal (FAILED)                     : 0
Dilewati (SYNTAX ERROR)            : 0
Success Rate                       : 100.00%

---------------------------------------------------------------------------
Report Excel  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\praktikum\07_AST_report.xlsx
Dataset JSON  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07_AST

Samples Data Sukses:


,nim,modul,file,kedalaman_ast,jumlah_node,jumlah_function,jumlah_variabel_unik,cyclomatic_complexity,loc,status,keterangan
0,MHS001,js02,MHS001_js02_p02,9,113,0,1,1,12,SUCCESS,
1,MHS001,js04,MHS001_js04_p01,7,360,0,12,3,38,SUCCESS,
2,MHS001,js04,MHS001_js04_p03,10,612,0,14,9,80,SUCCESS,
3,MHS001,js13,MHS001_js13_p03,10,472,0,14,1,65,SUCCESS,
4,MHS002,js02,MHS002_js02_p03,5,123,0,4,1,18,SUCCESS,
5,MHS002,js02,MHS002_js02_p04,5,142,0,4,1,18,SUCCESS,
6,MHS002,js05,MHS002_js05_p01,12,927,1,31,15,78,SUCCESS,
7,MHS002,js06,MHS002_js06_p01,7,341,0,16,1,49,SUCCESS,
8,MHS002,js07,MHS002_js07_p01,8,631,0,21,3,59,SUCCESS,
9,MHS002,js07,MHS002_js07_p04,8,443,0,16,2,47,SUCCESS,


Diproses pada 2026-06-11 22:49:18


In [13]:
# EXECUTION TUGAS EVAL
# --------------------------
print("="*60)
print(f"{' EXTRACT AST EVALUASI TUGAS':^60}")
print("-"*60)

INPUT_DIR = EVAL_T['SAMPLE']
OUTPUT_DIR = EVAL_T['AST']
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"{'Dataset input':<30}: {INPUT_DIR}")

# ambil semua file
all_py_files = collect_py_files(INPUT_DIR)
print(f"{'Total File input':<30}: {len(all_py_files)}")

logs = []
success = 0
failed = 0
syntax_err = 0

extract_start = start_timer()
total_loc = 0

for file_path in tqdm(all_py_files, desc="Extracting AST", unit="file"):
    log = process_file(file_path, INPUT_DIR, OUTPUT_DIR)
    logs.append(log)

    # update counter
    if log["status"] == "SUCCESS":
        success += 1
        if "loc" in log:
            total_loc += log["loc"]
    elif log["status"] == "FAILED":
        failed += 1
    elif log["status"] == "SYNTAX_ERROR":
        syntax_err += 1

# 1. Konversi logs ke DataFrame
df_logs = pd.DataFrame(logs)

print(f"{'AST disimpan di':<30}: {OUTPUT_DIR}")
total_mhs = count_students(df_logs)
total_files = count_all_files(OUTPUT_DIR)['total_files']
extract_time = save_execution_time(extract_start, "7_Ekstraksi_AST_EVAL_Tugas", total_mahasiswa=total_mhs, total_file=total_files, total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# 2. Filter data, ambil yang statusnya sukses
df_success = df_logs[df_logs['status'] == 'SUCCESS']
# 3. Simpan Report ke Excel
report_path = RESULTS_EVAL_T['EXTRACT_AST']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

# 4. Perhitungan Ringkasan Dasar (Logic Corrected)
total_input = len(all_py_files) # Total file .py di folder input
total_processed = len(df_logs)  # Total file yang masuk ke loop processing

# Success Rate dihitung dari (SUCCESS / TOTAL INPUT)
success_rate = (success / total_input * 100) if total_input > 0 else 0

display(HTML(f"<h3>RINGKASAN HASIL EKSTRAKSI AST</h3>"))
print("=" * 75)
print(f"{'Dataset Input':<35}: {INPUT_DIR}")
print(f"{'Total Mahasiswa':<35}: {total_mhs}")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total File Diproses':<35}: {total_processed}")
print(f"{'Waktu Eksekusi':<35}: {extract_time} seconds")
print(f"{'Total File JSON disimpan':<35}: {count_all_files(OUTPUT_DIR, exclude_dirs=['eval'])['total_files']}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal (FAILED)':<35}: {failed}")
print(f"{'Dilewati (SYNTAX ERROR)':<35}: {syntax_err}")
print(f"{'Success Rate':<35}: {success_rate:.2f}%")

print("\n" + "-" * 75)
print(f"Report Excel  : {report_path}")
print(f"Dataset JSON  : {OUTPUT_DIR}")

# Menampilkan 10 sampel teratas yang sukses
if not df_success.empty:
    print("\nSamples Data Sukses:")
    display(df_success.head(10))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                 EXTRACT AST EVALUASI TUGAS                 
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE
Total File input              : 850


Extracting AST:   0%|          | 0/850 [00:00<?, ?file/s]

AST disimpan di               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\07_AST

[SELESAI] 7_Ekstraksi_AST_EVAL_Tugas | 44.71 s | 850 file


Dataset Input                      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE
Total Mahasiswa                    : 50
Total File Input                   : 850
Total File Diproses                : 850
Waktu Eksekusi                     : 44.71 seconds
Total File JSON disimpan           : 850
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 850
Gagal (FAILED)                     : 0
Dilewati (SYNTAX ERROR)            : 0
Success Rate                       : 100.00%

---------------------------------------------------------------------------
Report Excel  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\tugas\07_AST_report.xlsx
Dataset JSON  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\07_AST

Samples Data Sukses:


,nim,modul,file,kedalaman_ast,jumlah_node,jumlah_function,jumlah_variabel_unik,cyclomatic_complexity,loc,status,keterangan
0,MHS001,js03,MHS001_js03_tp,11,1216,0,19,9,124,SUCCESS,
1,MHS001,js14,MHS001_js14_tp,8,1952,0,28,7,250,SUCCESS,
2,MHS002,js02,MHS002_js02_tp,9,248,0,6,3,32,SUCCESS,
3,MHS002,js03,MHS002_js03_tp,6,215,0,11,1,26,SUCCESS,
4,MHS002,js11,MHS002_js11_tp,9,1052,5,49,11,137,SUCCESS,
5,MHS002,js14,MHS002_js14_tp,8,227,0,8,1,27,SUCCESS,
6,MHS003,js02,MHS003_js02_tp,7,164,0,8,1,21,SUCCESS,
7,MHS004,js05,MHS004_js05_tp,6,292,0,10,2,40,SUCCESS,
8,MHS004,js08,MHS004_js08_tp,11,1290,0,52,26,171,SUCCESS,
9,MHS004,js14,MHS004_js14_tp,9,1426,0,10,3,152,SUCCESS,


Diproses pada 2026-06-11 22:50:04


### a. Visualisasi AST

In [ ]:
import os
import sys
import json
import ast
import traceback
import warnings
import logging
from tqdm.notebook import tqdm  # PERUBAHAN PENTING: Menggunakan versi auto/notebook untuk Jupyter
from datetime import datetime

# ===== PENCEGAHAN ANIMASI LOADING PECAH ==============================
# 1. Matikan peringatan standar
warnings.filterwarnings("ignore", category=UserWarning)

# 2. Matplotlib membandel karena menggunakan modul 'logging' C-level untuk peringatan Font.
# Kita matikan paksa akses loggernya di sini agar tidak merusak progress bar.
logging.getLogger('matplotlib.font_manager').disabled = True
logging.getLogger('matplotlib').setLevel(logging.ERROR)


# ===== IMPORT DEPENDENCIES ===========================================
try:
    import graphviz
    GRAPHVIZ_AVAILABLE = True
    print("✓ Graphviz tersedia")
except ImportError:
    GRAPHVIZ_AVAILABLE = False
    print("⚠️  Graphviz tidak tersedia, fallback ke matplotlib")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
    print("✓ Matplotlib tersedia")
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("⚠️  Matplotlib tidak tersedia")

# ===== KONFIGURASI =================================================
MAX_NODES_VIS = 80
MAX_DEPTH_VIS = 10

# Pastikan variabel DIRS sudah ada di cell environment Anda sebelumnya
INPUT_DIR      = DIRS['AST']
OUTPUT_VIS_DIR = DIRS['AST_VISUAL']
if os.path.exists(OUTPUT_VIS_DIR):
    shutil.rmtree(OUTPUT_VIS_DIR)
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

# ===== KUMPULKAN SEMUA .json =========================================
print("\n" + "="*70)
print("VALIDASI: VISUALISASI AST")
print("="*70)

all_ast_files = []
for nim in os.listdir(INPUT_DIR):
    nim_path = os.path.join(INPUT_DIR, nim)
    if not os.path.isdir(nim_path):
        continue
    for modul in os.listdir(nim_path):
        modul_path = os.path.join(nim_path, modul)
        if not os.path.isdir(modul_path):
            continue
        for fname in os.listdir(modul_path):
            if fname.endswith('.json'):
                fpath = os.path.join(modul_path, fname)
                all_ast_files.append((fpath, nim, modul, fname.replace('.json', '.py')))

print(f"✓ Input folder  : {INPUT_DIR}")
print(f"✓ Output folder : {OUTPUT_VIS_DIR}")
print(f"✓ Total AST JSON: {len(all_ast_files)} file")
print("="*70)

if len(all_ast_files) == 0:
    print("⚠️  Tidak ada file AST ditemukan!")
    print("   Jalankan cell AST extraction terlebih dahulu!")
    sys.exit()
else:
    print(f"✓ Siap visualisasi {len(all_ast_files)} file AST\n")


# ===== KONVERSI DICT → ast.AST OBJECT ================================
def is_valid_identifier(name):
    if not isinstance(name, str):
        return False
    if not name.isidentifier():
        return False
    if name.startswith('_'):
        return True
    return True

def dict_to_ast(d, path="root"):
    if not isinstance(d, dict):
        return None

    node_type_name = d.get('type')
    if not node_type_name or not isinstance(node_type_name, str):
        return None

    node_class = getattr(ast, node_type_name, None)
    if node_class is None or not (isinstance(node_class, type) and issubclass(node_class, ast.AST)):
        node = ast.AST()
        node._type_name = node_type_name
    else:
        node = node_class()

    for key, val in d.items():
        if key == 'type':
            continue

        if not isinstance(key, str) or not key.isidentifier():
            continue

        try:
            if isinstance(val, dict):
                child = dict_to_ast(val, f"{path}.{key}")
                if child is not None:
                    setattr(node, key, child)
            elif isinstance(val, list):
                converted = []
                for i, item in enumerate(val):
                    if isinstance(item, dict):
                        child = dict_to_ast(item, f"{path}.{key}[{i}]")
                        if child is not None:
                            converted.append(child)
                    elif not isinstance(item, dict):
                        converted.append(item)
                if converted:
                    setattr(node, key, converted)
            else:
                if not isinstance(val, dict):
                    setattr(node, key, val)
        except (AttributeError, TypeError):
            continue

    return node

def iter_children(node):
    if not isinstance(node, ast.AST):
        return
    try:
        for field, value in ast.iter_fields(node):
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, ast.AST):
                        yield item
            elif isinstance(value, ast.AST):
                yield value
    except Exception:
        pass


# ===== COLOR MAP & HELPERS ===========================================
COLOR_MAP = {
    'Module': '#4A90D9', 'FunctionDef': '#E94B3C', 'AsyncFunctionDef': '#E94B3C',
    'ClassDef': '#F5A623', 'If': '#7ED321', 'For': '#7ED321', 'AsyncFor': '#7ED321',
    'While': '#7ED321', 'Return': '#9B59B6', 'Call': '#1ABC9C', 'Assign': '#3498DB',
    'AugAssign': '#3498DB', 'AnnAssign': '#3498DB', 'Name': '#BDC3C7',
    'Constant': '#ECF0F1', 'Import': '#E67E22', 'ImportFrom': '#E67E22',
    'Expr': '#95A5A6', 'Compare': '#16A085', 'BoolOp': '#27AE60', 'BinOp': '#2980B9',
    'Attribute': '#8E44AD', 'Subscript': '#C0392B', 'List': '#D35400',
    'Dict': '#D35400', 'Tuple': '#D35400', 'Try': '#F39C12', 'With': '#1ABC9C',
    'Delete': '#E74C3C', 'Global': '#7F8C8D', 'Nonlocal': '#7F8C8D',
    'Pass': '#BDC3C7', 'Break': '#E74C3C', 'Continue': '#E74C3C',
    'Raise': '#C0392B', 'Assert': '#F39C12', 'Yield': '#9B59B6',
    'YieldFrom': '#9B59B6', 'Lambda': '#E94B3C', 'ExceptHandler': '#F39C12',
    'alias': '#E67E22', 'arguments': '#95A5A6', 'keyword': '#95A5A6',
}
DEFAULT_COLOR = '#DDEEFF'
LIGHT_NODES = {'#BDC3C7', '#ECF0F1', '#DDEEFF', '#95A5A6', '#7F8C8D'}

def _node_label(node):
    try:
        node_type = getattr(node, '_type_name', None) or type(node).__name__
        label = node_type
        name = getattr(node, 'name', None)
        nid = getattr(node, 'id', None)
        attr = getattr(node, 'attr', None)

        if node_type == 'alias':
            n = getattr(node, 'name', '')
            a = getattr(node, 'asname', '')
            label = f"alias\n{n}" + (f" as {a}" if a else "")
        elif node_type in ('Import', 'ImportFrom'):
            mod = getattr(node, 'module', '')
            label = f"{node_type}" + (f"\n{mod}" if mod else "")
        elif name and isinstance(name, str):
            label = f"{node_type}\n{str(name)[:14]}"
        elif nid and isinstance(nid, str):
            label = f"{node_type}\n{str(nid)[:14]}"
        elif attr and isinstance(attr, str):
            label = f"{node_type}\n.{str(attr)[:12]}"
        elif node_type == 'Constant':
            raw = getattr(node, 'value', '')
            label = f"Const\n{str(raw)[:12]}"
        elif node_type == 'BinOp':
            op_node = getattr(node, 'op', None)
            op_name = type(op_node).__name__ if op_node else ''
            label = f"BinOp\n{op_name}"
        elif node_type == 'BoolOp':
            op_node = getattr(node, 'op', None)
            op_name = type(op_node).__name__ if op_node else ''
            label = f"BoolOp\n{op_name}"
        elif node_type == 'Compare':
            ops = getattr(node, 'ops', [])
            ops_str = ','.join(type(o).__name__ for o in ops[:2]) if ops else ''
            label = f"Cmp\n{ops_str}"
        return label
    except Exception:
        return type(node).__name__


# ===== FUNGSI VISUALISASI ============================================
def visualize_ast_matplotlib(tree, nim, modul, filename, output_dir, max_nodes=MAX_NODES_VIS, max_depth=MAX_DEPTH_VIS):
    try:
        if not MATPLOTLIB_AVAILABLE:
            raise ImportError("Matplotlib tidak tersedia")

        positions = {}
        labels_map = {}
        node_types = {}
        edges = []
        counter = [0]
        node_count = [0]
        x_counter = [0]

        def build_graph(node, parent_id=None, depth=0):
            if node_count[0] >= max_nodes or depth > max_depth:
                return None
            if not isinstance(node, ast.AST):
                return None

            node_type = getattr(node, '_type_name', None) or type(node).__name__
            current_id = counter[0]
            counter[0] += 1
            node_count[0] += 1

            label = _node_label(node)
            children = list(iter_children(node))
            children_ids = []

            for child in children:
                cid = build_graph(child, current_id, depth + 1)
                if cid is not None:
                    children_ids.append(cid)
                    edges.append((current_id, cid))

            if children_ids:
                xs = [positions[cid][0] for cid in children_ids]
                my_x = (min(xs) + max(xs)) / 2
            else:
                my_x = x_counter[0]
                x_counter[0] += 1

            positions[current_id] = (my_x, -depth)
            labels_map[current_id] = label
            node_types[current_id] = node_type
            return current_id

        build_graph(tree)

        if not positions:
            return None, "Tidak ada node yang berhasil di-layout"

        n_leaves = max(x_counter[0], 1)
        fig_w = max(14, n_leaves * 0.9)
        fig, ax = plt.subplots(figsize=(fig_w, 10))
        ax.set_facecolor('#F8F9FA')
        fig.patch.set_facecolor('#F8F9FA')

        for (src, dst) in edges:
            x1, y1 = positions[src]
            x2, y2 = positions[dst]
            ax.plot([x1, x2], [y1, y2], color='#AAAAAA', lw=0.6, alpha=0.6, zorder=1)

        for nid, (x, y) in positions.items():
            node_type = node_types[nid]
            color = COLOR_MAP.get(node_type, DEFAULT_COLOR)
            fontcolor = '#FFFFFF' if color not in LIGHT_NODES else '#333333'
            ax.scatter(x, y, s=320, c=color, zorder=3, edgecolors='#334455', linewidths=0.7)
            ax.text(x, y, labels_map[nid], ha='center', va='center', fontsize=5.5, color=fontcolor, zorder=4, multialignment='center')

        total_nodes = node_count[0]
        ax.set_title(f"AST Tree — {nim} / {modul} / {filename}  [{total_nodes} node ditampilkan]", fontsize=10, fontweight='bold', pad=8)
        ax.axis('off')
        plt.tight_layout()

        out_subdir = os.path.join(output_dir, nim, modul)
        os.makedirs(out_subdir, exist_ok=True)
        out_path = os.path.join(out_subdir, filename.replace('.py', '.png'))
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        return out_path, None
    except Exception as e:
        return None, traceback.format_exc()

def visualize_ast_graphviz(tree, nim, modul, filename, output_dir, max_nodes=MAX_NODES_VIS, max_depth=MAX_DEPTH_VIS):
    try:
        if not GRAPHVIZ_AVAILABLE:
            raise ImportError("Graphviz tidak tersedia")
        
        import graphviz
        dot = graphviz.Digraph(comment=f'AST {nim}/{modul}/{filename}')
        dot.attr(rankdir='TB', size='18,14', dpi='150')
        dot.attr('node', shape='box', style='filled', fontname='Helvetica', fontsize='9', width='0.5', height='0.35')

        counter = [0]
        node_count = [0]

        def add_node(node, parent_id=None, depth=0):
            if node_count[0] >= max_nodes or depth > max_depth:
                return
            if not isinstance(node, ast.AST):
                return

            node_type = getattr(node, '_type_name', None) or type(node).__name__
            current_id = str(counter[0])
            counter[0] += 1
            node_count[0] += 1

            label = _node_label(node)
            color = COLOR_MAP.get(node_type, DEFAULT_COLOR)
            fontcolor = '#FFFFFF' if color not in LIGHT_NODES else '#333333'

            dot.node(current_id, label=label, fillcolor=color, fontcolor=fontcolor)
            if parent_id is not None:
                dot.edge(parent_id, current_id)

            for child in iter_children(node):
                add_node(child, current_id, depth + 1)

        add_node(tree)

        out_subdir = os.path.join(output_dir, nim, modul)
        os.makedirs(out_subdir, exist_ok=True)
        out_base = os.path.join(out_subdir, filename.replace('.py', ''))
        dot.render(out_base, format='png', cleanup=True)
        return out_base + '.png', None
    except Exception as e:
        return None, traceback.format_exc()

def visualize_ast(tree, nim, modul, filename, output_dir):
    if GRAPHVIZ_AVAILABLE:
        out_path, err = visualize_ast_graphviz(tree, nim, modul, filename, output_dir)
        if err is None:
            return out_path, None
        if MATPLOTLIB_AVAILABLE:
            return visualize_ast_matplotlib(tree, nim, modul, filename, output_dir)
        else:
            return None, f"Graphviz gagal dan Matplotlib tidak tersedia: {err}"
    elif MATPLOTLIB_AVAILABLE:
        return visualize_ast_matplotlib(tree, nim, modul, filename, output_dir)
    else:
        return None, "Graphviz dan Matplotlib tidak tersedia"


# ===== EKSEKUSI UTAMA ================================================
if len(all_ast_files) > 0:
    vis_success = 0
    vis_failed  = 0
    vis_skipped = 0
    
    visual_start = start_timer()

    print(f"{'VISUALISASI AST (BACA JSON → PNG)':^70}")
    print("=" * 70)
    
    # tqdm.auto akan merender progress bar di atas (sebagai UI/HTML widget di Jupyter) 
    # dan log error yang dicetak akan berjejer rapi ke bawah.
    for ast_json_path, nim, modul, filename in tqdm(all_ast_files, desc="Visualisasi AST", unit="file"):
        file_id = os.path.splitext(filename)[0]
        try:
            with open(ast_json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            ast_dict = data.get('ast')
            if not ast_dict or not isinstance(ast_dict, dict):
                vis_skipped += 1
                continue

            tree = dict_to_ast(ast_dict)
            if tree is None:
                vis_skipped += 1
                continue

            out_path, err = visualize_ast(tree, nim, modul, filename, OUTPUT_VIS_DIR)
            if err:
                vis_failed += 1
                # Cetak log menggunakan standard print atau tqdm.write
                print(f"❌ [GAGAL] {nim}/{modul}/{filename}\n   └─ Error: {err[:150]}...")
            else:
                vis_success += 1

        except json.JSONDecodeError as e:
            vis_failed += 1
            print(f"❌ [GAGAL JSON] {nim}/{modul}/{filename}\n   └─ Error: {str(e)[:150]}")
        except Exception as e:
            vis_failed += 1
            print(f"❌ [GAGAL EXCEPTION] {nim}/{modul}/{filename}\n   └─ Error: {str(e)[:150]}")

    visual_total_time = save_execution_time(visual_start, "7_a_Visualisasi_AST", total_mahasiswa=count_students(DIRS['AST_VISUAL']), total_file={len(all_ast_files)}, total_size_kb=get_path_size(OUTPUT_VIS_DIR))
    
    
    display(HTML(f"<h3>RINGKASAN HASIL VISUALISASI AST</h3>"))
    print(f"{'Waktu Eksekusi':<10}: {visual_total_time} seconds")
    print(f"{'Berhasil':<10}: {vis_success}")
    print(f"{'Gagal':<10}: {vis_failed}")
    print(f"{'Skip':<10}: {vis_skipped}")
    print(f"{'Total':<10}: {len(all_ast_files)}")
    print(f"{'Output':<10}: {OUTPUT_VIS_DIR}")
    print("=" * 70)

else:
    print("⚠️  Tidak ada file AST untuk divisualisasi")

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Proses visualisasi selesai pada {timestamp}")

⚠️  Graphviz tidak tersedia, fallback ke matplotlib
✓ Matplotlib tersedia

VALIDASI: VISUALISASI AST
✓ Input folder  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST
✓ Output folder : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07a_AST_visual
✓ Total AST JSON: 2321 file
✓ Siap visualisasi 2321 file AST

                  VISUALISASI AST (BACA JSON → PNG)                   


Visualisasi AST:   0%|          | 0/2321 [00:00<?, ?file/s]

Waktu Eksekusi: 3034.49 seconds
Berhasil  : 2321
Gagal     : 0
Skip      : 0
Total     : 2321
Output    : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07a_AST_visual
Proses visualisasi selesai pada 2026-06-02 02:01:19


## 8. Transformasi AST ke Graph

In [14]:
# FUNCTION HELPER - GRAPH CONSTRUCTION (PICKLE VERSION)
# ==============================================================================

def save_log(nim, modul, file, n_nodes=0, n_edges=0, size=0, status="SUCCESS", message=""):
    return {
        "nim": nim, "modul": modul, "file": file,
        "jumlah_node_graf": n_nodes, "jumlah_edge_graf": n_edges,
        "size_kb" : size,
        "status": status, "keterangan": message
    }

def get_rich_label(ast_node):
    node_type = ast_node.get("type", "Unknown")
    
    # 1. Operasi Biner (misal: Add, Sub, Mult)
    if node_type == "BinOp" and "op" in ast_node and isinstance(ast_node["op"], dict):
        return f"BinOp_{ast_node['op'].get('type', '')}"
    # 2. Operasi Perbandingan (misal: Eq, Gt, Lt)
    elif node_type == "Compare" and "ops" in ast_node and isinstance(ast_node["ops"], list):
        ops_labels = [op.get("type", "") for op in ast_node["ops"] if isinstance(op, dict)]
        if ops_labels:
            return f"Compare_{'_'.join(ops_labels)}"            
    # 3. Operasi Boolean (misal: And, Or)
    elif node_type == "BoolOp" and "op" in ast_node and isinstance(ast_node["op"], dict):
        return f"BoolOp_{ast_node['op'].get('type', '')}"
    # 4. Tipe Konstanta (membedakan int, float, str tanpa mengambil nilainya)
    elif node_type == "Constant" and "value" in ast_node:
        val_type = type(ast_node["value"]).__name__
        return f"Constant_{val_type}"
    # Jika tidak masuk kategori di atas, kembalikan tipe aslinya
    return node_type

def transform_ast_to_nx(ast_node, G=None, parent_id=None):
    if G is None: 
        G = nx.Graph()
        
    current_id = G.number_of_nodes()
    
    # Ambil tipe AST (misal: 'For', 'Assign', 'BinOp')
    raw_feature = get_rich_label(ast_node)
    node_feature = str(raw_feature)
    
    # Masukkan ke graf menggunakan key 'feature'
    G.add_node(current_id, feature=node_feature, lineno=ast_node.get("lineno"), end_lineno=ast_node.get("end_lineno"))
    
    if parent_id is not None: 
        G.add_edge(parent_id, current_id)
        
    for field, value in ast_node.items():
        if field == "type": continue
        
        if field in ["op", "ops"] and ast_node.get("type") in ["BinOp", "Compare", "BoolOp", "UnaryOp"]:
            continue
        
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict) and "type" in item:
                    transform_ast_to_nx(item, G, current_id)
        elif isinstance(value, dict) and "type" in value:
            transform_ast_to_nx(value, G, current_id)
            
    return G

# SAVE GRAPH AS PICKLE
def save_graph_pkl(G, metadata, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "wb") as f:
        pickle.dump({"metadata": metadata, "graph": G}, f)
        
def normalize_graph(G):
    return nx.convert_node_labels_to_integers(G, first_label=0)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Diproses pada {timestamp}")

✅ Diproses pada 2026-06-11 22:50:04


In [15]:
# RUN & EXECUTION PRAKTIKUM
# ==============================================================================
INPUT_PY_DIR = DIRS['FILTERED_PRAK'] # Folder berisi file .py hasil filter requirement
INPUT_DIR = DIRS['AST_P']   # Folder JSON AST
OUTPUT_DIR = DIRS['GRAPH_P'] # Folder baru untuk Dataset Graph (.pkl per file)

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mengumpulkan file JSON AST
all_ast_files = []
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".json"):
            all_ast_files.append(os.path.join(root, file))

logs = []
success = 0
failed = 0

graph_start = start_timer()

print("="*60)
print(f"{' CONSTRUCT GRAPH DATASET PRAKTIKUM':^60}")
print("-"*60)
print(f"{'Dataset input':<30}: {INPUT_DIR}")
print(f"{'Total File input':<30}: {len(all_ast_files)}")

for file_path in tqdm(all_ast_files, desc="Transforming to Graph", unit="file"):
    # 1. Extract Metadata
    rel_path = os.path.relpath(file_path, INPUT_DIR)
    parts = rel_path.split(os.sep)
    nim, modul, filename = parts[0], parts[1], os.path.basename(file_path)
    file_id = os.path.splitext(filename)[0]
    
    try:
        # 2. Load AST & Transform
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        metrics = data.get("metrics", {})
        G = transform_ast_to_nx(data['ast'])
        G = normalize_graph(G)
        
        # 3. Save as individual .pkl
        output_path = os.path.join(OUTPUT_DIR, rel_path.replace(".json", ".pkl"))
        meta = {
            "nim": nim,
            "modul": modul,
            "file": data['filename'],
            "source_path": os.path.join(
                INPUT_PY_DIR,
                nim,
                modul,
                f"{os.path.splitext(filename)[0]}.py"
            ),
            "jumlah_nodes": G.number_of_nodes(),
            "jumlah_edge": G.number_of_edges(),
            
            "kedalaman_ast": metrics.get("kedalaman_ast", 0),
            "jumlah_function": metrics.get("jumlah_function", 0),
            "jumlah_variabel_unik": metrics.get("jumlah_variabel_unik", 0),
            "loc": metrics.get("loc", 0)
        }
        save_graph_pkl(G, meta, output_path)
        
        graph_size = get_path_size(output_path, unit='kb')
        
        logs.append(save_log(nim, modul, filename, G.number_of_nodes(), G.number_of_edges(), graph_size))
        success += 1
        
    except Exception as e:
        logs.append(save_log(nim, modul, filename, status="FAILED", message=str(e)))
        failed += 1
        
df_logs = pd.DataFrame(logs)
print(f"\nDataset Graph (Pickle) disimpan di: {OUTPUT_DIR}")
execution_time = save_execution_time(graph_start, "8_Construct_Graph_praktikum", total_mahasiswa=count_students(OUTPUT_DIR), total_file=count_all_files(OUTPUT_DIR)['total_files'], total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# ==============================================================================
df_success = df_logs[df_logs['status'] == 'SUCCESS']

# Simpan Report
report_path = RESULTS['CONSTRUCT_GRAPH_P']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

total_input = len(all_ast_files)
total_processed = len(df_logs)

display(HTML(f"<h3>RINGKASAN HASIL KONSTRUKSI GRAPH</h3>"))
print("-" * 75)
print(f"{'Waktu Eksekusi':<35}: {execution_time} seconds")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total Diproses':<35}: {total_processed}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal':<35}: {failed}")
print(f"{'Success Rate':<35}: {(success/len(all_ast_files)*100):.2f}%")
print("-" * 75)
print(f"{'Report Excel':<25}: {report_path}")
print(f"{'Dataset Graph':<25}: {OUTPUT_DIR}")

# Preview
print("\nPreview Data (10 baris pertama):")
display(df_logs.head(10))

# Timestamp
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

              CONSTRUCT GRAPH DATASET PRAKTIKUM             
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST_prak
Total File input              : 1678


Transforming to Graph:   0%|          | 0/1678 [00:00<?, ?file/s]


Dataset Graph (Pickle) disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph_prak

[SELESAI] 8_Construct_Graph_praktikum | 63.81 s | 1678 file


---------------------------------------------------------------------------
Waktu Eksekusi                     : 63.81 seconds
Total File Input                   : 1678
Total Diproses                     : 1678
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 1678
Gagal                              : 0
Success Rate                       : 100.00%
---------------------------------------------------------------------------
Report Excel             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\08_Graph_report_prak.xlsx
Dataset Graph            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph_prak

Preview Data (10 baris pertama):


,nim,modul,file,jumlah_node_graf,jumlah_edge_graf,size_kb,status,keterangan
0,MHS001,js02,MHS001_js02_p01.json,414,413,17.56,SUCCESS,
1,MHS001,js02,MHS001_js02_p02.json,113,112,5.05,SUCCESS,
2,MHS001,js02,MHS001_js02_p03.json,212,211,8.93,SUCCESS,
3,MHS001,js02,MHS001_js02_p04.json,142,141,6.11,SUCCESS,
4,MHS001,js03,MHS001_js03_p01.json,498,497,21.72,SUCCESS,
5,MHS001,js03,MHS001_js03_p02.json,142,141,6.11,SUCCESS,
6,MHS001,js03,MHS001_js03_p03.json,224,223,9.46,SUCCESS,
7,MHS001,js03,MHS001_js03_p04.json,53,52,2.43,SUCCESS,
8,MHS001,js04,MHS001_js04_p01.json,358,357,15.12,SUCCESS,
9,MHS001,js04,MHS001_js04_p02.json,1577,1576,68.24,SUCCESS,


Diproses pada 2026-06-11 22:51:10


In [16]:
# RUN & EXECUTION TUGAS
# ==============================================================================
INPUT_PY_DIR = DIRS['FILTERED_TGS'] # Folder berisi file .py hasil filter requirement
INPUT_DIR = DIRS['AST_T']   # Folder JSON AST
OUTPUT_DIR = DIRS['GRAPH_T'] # Folder baru untuk Dataset Graph (.pkl per file)

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mengumpulkan file JSON AST
all_ast_files = []
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".json"):
            all_ast_files.append(os.path.join(root, file))

logs = []
success = 0
failed = 0

graph_start = start_timer()

print("="*60)
print(f"{' CONSTRUCT GRAPH DATASET TUGAS ':^60}")
print("-"*60)
print(f"{'Dataset input':<30}: {INPUT_DIR}")
print(f"{'Total File input':<30}: {len(all_ast_files)}")

for file_path in tqdm(all_ast_files, desc="Transforming to Graph", unit="file"):
    # 1. Extract Metadata
    rel_path = os.path.relpath(file_path, INPUT_DIR)
    parts = rel_path.split(os.sep)
    nim, modul, filename = parts[0], parts[1], os.path.basename(file_path)
    file_id = os.path.splitext(filename)[0]
    
    try:
        # 2. Load AST & Transform
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        metrics = data.get("metrics", {})
        G = transform_ast_to_nx(data['ast'])
        G = normalize_graph(G)
        
        # 3. Save as individual .pkl
        output_path = os.path.join(OUTPUT_DIR, rel_path.replace(".json", ".pkl"))
        meta = {
            "nim": nim,
            "modul": modul,
            "file": data['filename'],
            "source_path": os.path.join(
                INPUT_PY_DIR,
                nim,
                modul,
                f"{os.path.splitext(filename)[0]}.py"
            ),
            "jumlah_nodes": G.number_of_nodes(),
            "jumlah_edge": G.number_of_edges(),
            
            "kedalaman_ast": metrics.get("kedalaman_ast", 0),
            "jumlah_function": metrics.get("jumlah_function", 0),
            "jumlah_variabel_unik": metrics.get("jumlah_variabel_unik", 0),
            "loc": metrics.get("loc", 0)
        }
        save_graph_pkl(G, meta, output_path)
        
        graph_size = get_path_size(output_path, unit='kb')
        
        logs.append(save_log(nim, modul, filename, G.number_of_nodes(), G.number_of_edges(), graph_size))
        success += 1
        
    except Exception as e:
        logs.append(save_log(nim, modul, filename, status="FAILED", message=str(e)))
        failed += 1
        
df_logs = pd.DataFrame(logs)
print(f"\nDataset Graph (Pickle) disimpan di: {OUTPUT_DIR}")
execution_time = save_execution_time(graph_start, "8_Construct_Graph_tugas", total_mahasiswa=count_students(OUTPUT_DIR), total_file=count_all_files(OUTPUT_DIR)['total_files'], total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# ==============================================================================
df_success = df_logs[df_logs['status'] == 'SUCCESS']

# Simpan Report
report_path = RESULTS['CONSTRUCT_GRAPH_T']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

total_input = len(all_ast_files)
total_processed = len(df_logs)

display(HTML(f"<h3>RINGKASAN HASIL KONSTRUKSI GRAPH</h3>"))
print("-" * 75)
print(f"{'Waktu Eksekusi':<35}: {execution_time} seconds")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total Diproses':<35}: {total_processed}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal':<35}: {failed}")
print(f"{'Success Rate':<35}: {(success/len(all_ast_files)*100):.2f}%")
print("-" * 75)
print(f"{'Report Excel':<25}: {report_path}")
print(f"{'Dataset Graph':<25}: {OUTPUT_DIR}")

# Preview
print("\nPreview Data (10 baris pertama):")
display(df_logs.head(10))

# Timestamp
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

               CONSTRUCT GRAPH DATASET TUGAS                
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_AST_tgs
Total File input              : 566


Transforming to Graph:   0%|          | 0/566 [00:00<?, ?file/s]


Dataset Graph (Pickle) disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph_tgs

[SELESAI] 8_Construct_Graph_tugas | 22.65 s | 566 file


---------------------------------------------------------------------------
Waktu Eksekusi                     : 22.65 seconds
Total File Input                   : 566
Total Diproses                     : 566
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 566
Gagal                              : 0
Success Rate                       : 100.00%
---------------------------------------------------------------------------
Report Excel             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\08_Graph_report_tgs.xlsx
Dataset Graph            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph_tgs

Preview Data (10 baris pertama):


,nim,modul,file,jumlah_node_graf,jumlah_edge_graf,size_kb,status,keterangan
0,MHS001,js02,MHS001_js02_tp.json,632,631,27.36,SUCCESS,
1,MHS001,js03,MHS001_js03_tp.json,1163,1162,51.17,SUCCESS,
2,MHS001,js04,MHS001_js04_tp.json,2748,2747,121.75,SUCCESS,
3,MHS001,js13,MHS001_js13_tp.json,2055,2054,91.15,SUCCESS,
4,MHS001,js14,MHS001_js14_tp.json,1918,1917,84.68,SUCCESS,
5,MHS002,js02,MHS002_js02_tp.json,246,245,10.29,SUCCESS,
6,MHS002,js03,MHS002_js03_tp.json,215,214,9.01,SUCCESS,
7,MHS002,js04,MHS002_js04_tp.json,1106,1105,47.93,SUCCESS,
8,MHS002,js05,MHS002_js05_tp.json,313,312,13.16,SUCCESS,
9,MHS002,js06,MHS002_js06_tp.json,538,537,22.94,SUCCESS,


Diproses pada 2026-06-11 22:51:34


In [17]:
# RUN & EXECUTION EVAL PRAKTIKUM
# ==============================================================================
INPUT_PY_DIR = EVAL_P['SAMPLE'] # Folder berisi file .py hasil filter requirement
INPUT_DIR = EVAL_P['AST']   # Folder JSON AST
OUTPUT_DIR = EVAL_P['GRAPH'] # Folder baru untuk Dataset Graph (.pkl per file)

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mengumpulkan file JSON AST
all_ast_files = []
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".json"):
            all_ast_files.append(os.path.join(root, file))

logs = []
success = 0
failed = 0

graph_start = start_timer()

print("="*60)
print(f"{' CONSTRUCT GRAPH DATASET EVAL PRAKTIKUM ':^60}")
print("-"*60)
print(f"{'Dataset input':<30}: {INPUT_DIR}")
print(f"{'Total File input':<30}: {len(all_ast_files)}")

for file_path in tqdm(all_ast_files, desc="Transforming to Graph", unit="file"):
    # 1. Extract Metadata
    rel_path = os.path.relpath(file_path, INPUT_DIR)
    parts = rel_path.split(os.sep)
    nim, modul, filename = parts[0], parts[1], os.path.basename(file_path)
    file_id = os.path.splitext(filename)[0]
    
    try:
        # 2. Load AST & Transform
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        metrics = data.get("metrics", {})
        G = transform_ast_to_nx(data['ast'])
        G = normalize_graph(G)
        
        # 3. Save as individual .pkl
        output_path = os.path.join(OUTPUT_DIR, rel_path.replace(".json", ".pkl"))
        meta = {
            "nim": data['nim'],
            "modul": data['modul'],
            "file": data['filename'],
            "source_path": os.path.join(
                INPUT_PY_DIR,
                nim,
                f"{os.path.splitext(filename)[0]}.py"
            ),
            "jumlah_nodes": G.number_of_nodes(),
            "jumlah_edge": G.number_of_edges(),
            
            "kedalaman_ast": metrics.get("kedalaman_ast", 0),
            "jumlah_function": metrics.get("jumlah_function", 0),
            "jumlah_variabel_unik": metrics.get("jumlah_variabel_unik", 0),
            "loc": metrics.get("loc", 0)
        }
        save_graph_pkl(G, meta, output_path)
        
        graph_size = get_path_size(output_path, unit='kb')
        
        logs.append(save_log(data['nim'], data['modul'], filename, G.number_of_nodes(), G.number_of_edges(), graph_size))
        success += 1
        
    except Exception as e:
        logs.append(save_log(data['nim'], data['modul'], filename, status="FAILED", message=str(e)))
        failed += 1
        
df_logs = pd.DataFrame(logs)
print(f"\nDataset Graph (Pickle) disimpan di: {OUTPUT_DIR}")
execution_time = save_execution_time(graph_start, "8_Construct_Graph_EVAL_praktikum", total_mahasiswa=count_students(OUTPUT_DIR), total_file=count_all_files(OUTPUT_DIR)['total_files'], total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# ==============================================================================
df_success = df_logs[df_logs['status'] == 'SUCCESS']

# Simpan Report
report_path = RESULTS_EVAL_P['CONSTRUCT_GRAPH']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

total_input = len(all_ast_files)
total_processed = len(df_logs)

display(HTML(f"<h3>RINGKASAN HASIL KONSTRUKSI GRAPH</h3>"))
print("-" * 75)
print(f"{'Waktu Eksekusi':<35}: {execution_time} seconds")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total Diproses':<35}: {total_processed}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal':<35}: {failed}")
print(f"{'Success Rate':<35}: {(success/len(all_ast_files)*100):.2f}%")
print("-" * 75)
print(f"{'Report Excel':<25}: {report_path}")
print(f"{'Dataset Graph':<25}: {OUTPUT_DIR}")

# Preview
print("\nPreview Data (10 baris pertama):")
display(df_logs.head(10))

# Timestamp
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

           CONSTRUCT GRAPH DATASET EVAL PRAKTIKUM           
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07_AST
Total File input              : 2515


Transforming to Graph:   0%|          | 0/2515 [00:00<?, ?file/s]


Dataset Graph (Pickle) disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\08_Graph

[SELESAI] 8_Construct_Graph_EVAL_praktikum | 101.22 s | 2515 file


---------------------------------------------------------------------------
Waktu Eksekusi                     : 101.22 seconds
Total File Input                   : 2515
Total Diproses                     : 2515
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 2515
Gagal                              : 0
Success Rate                       : 100.00%
---------------------------------------------------------------------------
Report Excel             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\praktikum\08_Graph_report.xlsx
Dataset Graph            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\08_Graph

Preview Data (10 baris pertama):


,nim,modul,file,jumlah_node_graf,jumlah_edge_graf,size_kb,status,keterangan
0,MHS001,js02,MHS001_js02_p02.json,113,112,5.05,SUCCESS,
1,MHS001,js04,MHS001_js04_p01.json,358,357,15.12,SUCCESS,
2,MHS001,js04,MHS001_js04_p03.json,593,592,25.56,SUCCESS,
3,MHS001,js13,MHS001_js13_p03.json,472,471,20.16,SUCCESS,
4,MHS002,js02,MHS002_js02_p03.json,123,122,5.41,SUCCESS,
5,MHS002,js02,MHS002_js02_p04.json,142,141,6.11,SUCCESS,
6,MHS002,js05,MHS002_js05_p01.json,898,897,38.89,SUCCESS,
7,MHS002,js06,MHS002_js06_p01.json,338,337,14.17,SUCCESS,
8,MHS002,js07,MHS002_js07_p01.json,622,621,26.57,SUCCESS,
9,MHS002,js07,MHS002_js07_p04.json,437,436,18.46,SUCCESS,


Diproses pada 2026-06-11 22:53:17


In [18]:
# RUN & EXECUTION EVAL TUGAS
# ==============================================================================
INPUT_PY_DIR = EVAL_T['SAMPLE'] # Folder berisi file .py hasil filter requirement
INPUT_DIR = EVAL_T['AST']   # Folder JSON AST
OUTPUT_DIR = EVAL_T['GRAPH'] # Folder baru untuk Dataset Graph (.pkl per file)

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mengumpulkan file JSON AST
all_ast_files = []
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".json"):
            all_ast_files.append(os.path.join(root, file))

logs = []
success = 0
failed = 0

graph_start = start_timer()

print("="*60)
print(f"{' CONSTRUCT GRAPH DATASET EVAL TUGAS ':^60}")
print("-"*60)
print(f"{'Dataset input':<30}: {INPUT_DIR}")
print(f"{'Total File input':<30}: {len(all_ast_files)}")

for file_path in tqdm(all_ast_files, desc="Transforming to Graph", unit="file"):
    # 1. Extract Metadata
    rel_path = os.path.relpath(file_path, INPUT_DIR)
    parts = rel_path.split(os.sep)
    nim, modul, filename = parts[0], parts[1], os.path.basename(file_path)
    file_id = os.path.splitext(filename)[0]
    
    try:
        # 2. Load AST & Transform
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        metrics = data.get("metrics", {})
        G = transform_ast_to_nx(data['ast'])
        G = normalize_graph(G)
        
        # 3. Save as individual .pkl
        output_path = os.path.join(OUTPUT_DIR, rel_path.replace(".json", ".pkl"))
        meta = {
            "nim": data['nim'],
            "modul": data['modul'],
            "file": data['filename'],
            "source_path": os.path.join(
                INPUT_PY_DIR,
                nim,
                f"{os.path.splitext(filename)[0]}.py"
            ),
            "jumlah_nodes": G.number_of_nodes(),
            "jumlah_edge": G.number_of_edges(),
            
            "kedalaman_ast": metrics.get("kedalaman_ast", 0),
            "jumlah_function": metrics.get("jumlah_function", 0),
            "jumlah_variabel_unik": metrics.get("jumlah_variabel_unik", 0),
            "loc": metrics.get("loc", 0)
        }
        save_graph_pkl(G, meta, output_path)
        
        graph_size = get_path_size(output_path, unit='kb')
        
        logs.append(save_log(data['nim'], data['modul'], filename, G.number_of_nodes(), G.number_of_edges(), graph_size))
        success += 1
        
    except Exception as e:
        logs.append(save_log(data['nim'], data['modul'], filename, status="FAILED", message=str(e)))
        failed += 1
        
df_logs = pd.DataFrame(logs)
print(f"\nDataset Graph (Pickle) disimpan di: {OUTPUT_DIR}")
execution_time = save_execution_time(graph_start, "8_Construct_Graph_EVAL_tugas", total_mahasiswa=count_students(OUTPUT_DIR), total_file=count_all_files(OUTPUT_DIR)['total_files'], total_size_kb=get_path_size(OUTPUT_DIR))

# SUMMARY & REPORTING
# ==============================================================================
df_success = df_logs[df_logs['status'] == 'SUCCESS']

# Simpan Report
report_path = RESULTS_EVAL_T['CONSTRUCT_GRAPH']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

total_input = len(all_ast_files)
total_processed = len(df_logs)

display(HTML(f"<h3>RINGKASAN HASIL KONSTRUKSI GRAPH</h3>"))
print("-" * 75)
print(f"{'Waktu Eksekusi':<35}: {execution_time} seconds")
print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total Diproses':<35}: {total_processed}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal':<35}: {failed}")
print(f"{'Success Rate':<35}: {(success/len(all_ast_files)*100):.2f}%")
print("-" * 75)
print(f"{'Report Excel':<25}: {report_path}")
print(f"{'Dataset Graph':<25}: {OUTPUT_DIR}")

# Preview
print("\nPreview Data (10 baris pertama):")
display(df_logs.head(10))

# Timestamp
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

             CONSTRUCT GRAPH DATASET EVAL TUGAS             
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\07_AST
Total File input              : 850


Transforming to Graph:   0%|          | 0/850 [00:00<?, ?file/s]


Dataset Graph (Pickle) disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\08_Graph

[SELESAI] 8_Construct_Graph_EVAL_tugas | 36.84 s | 850 file


---------------------------------------------------------------------------
Waktu Eksekusi                     : 36.84 seconds
Total File Input                   : 850
Total Diproses                     : 850
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 850
Gagal                              : 0
Success Rate                       : 100.00%
---------------------------------------------------------------------------
Report Excel             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\tugas\08_Graph_report.xlsx
Dataset Graph            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\08_Graph

Preview Data (10 baris pertama):


,nim,modul,file,jumlah_node_graf,jumlah_edge_graf,size_kb,status,keterangan
0,MHS001,js03,MHS001_js03_tp.json,1163,1162,51.17,SUCCESS,
1,MHS001,js14,MHS001_js14_tp.json,1918,1917,84.68,SUCCESS,
2,MHS002,js02,MHS002_js02_tp.json,246,245,10.29,SUCCESS,
3,MHS002,js03,MHS002_js03_tp.json,215,214,9.01,SUCCESS,
4,MHS002,js11,MHS002_js11_tp.json,1040,1039,44.75,SUCCESS,
5,MHS002,js14,MHS002_js14_tp.json,223,222,9.53,SUCCESS,
6,MHS003,js02,MHS003_js02_tp.json,164,163,6.98,SUCCESS,
7,MHS004,js05,MHS004_js05_tp.json,286,285,12.08,SUCCESS,
8,MHS004,js08,MHS004_js08_tp.json,1259,1258,54.26,SUCCESS,
9,MHS004,js14,MHS004_js14_tp.json,1420,1419,61.74,SUCCESS,


Diproses pada 2026-06-11 22:53:54


## 9. Building List of Graph

In [19]:
# FUNCTION HELPER - GRAPH2VEC INPUT PREPARATION
# ==============================================================================

def collect_pkl_files(input_dir):
    """Mengumpulkan semua path file .pkl dari direktori dataset graf."""
    pkl_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".pkl"):
                pkl_files.append(os.path.join(root, file))
    return pkl_files

def normalize_graph_for_karateclub(G):
    """
    Normalisasi wajib untuk algoritma Graph2Vec (Karateclub):
    1. Memastikan graf tidak berarah (Undirected).
    2. Memastikan ID node adalah integer berurutan yang dimulai dari 0.
    """
    # 1. Konversi ke Undirected Graph jika masih berarah
    if G.is_directed():
        G = G.to_undirected()
        
    # 2. Reset ID node agar berurutan (0, 1, 2, ...) 
    # tanpa menghilangkan atribut 'feature' yang sudah kita buat
    G = nx.convert_node_labels_to_integers(G, first_label=0)
    
    return G

def load_single_graph_data(filepath):
    """Membaca satu file .pkl yang berisi graf dan metadatanya."""
    try:
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        return data.get("graph"), data.get("metadata"), "SUCCESS", ""
    except Exception as e:
        return None, None, "FAILED", str(e)

def save_log(meta, n_nodes=0, n_edges=0, status="SUCCESS", message=""):
    return {
        "nim": meta.get("nim") if meta else None,
        "modul": meta.get("modul") if meta else None,
        "file": meta.get("file") if meta else None,
        "jumlah_node": n_nodes,
        "jumlah_edge": n_edges,
        "group_key": f"{meta.get('modul')}_{meta.get('file')}" if meta else None,
        "status": status,
        "keterangan": message
    }

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Helper functions untuk persiapan input Graph2Vec dijalankan pada {timestamp}")

✅ Helper functions untuk persiapan input Graph2Vec dijalankan pada 2026-06-11 22:53:54


In [20]:
def build_list_of_graph(INPUT_DIR, OUTPUT_DATASET_PATH, report_path, stage_name, is_eval=False):
    # RUN & EXECUTION
    # ==============================================================================
    all_pkl_files = collect_pkl_files(INPUT_DIR)
    if os.path.exists(OUTPUT_DATASET_PATH):
        shutil.rmtree(OUTPUT_DATASET_PATH)
    os.makedirs(OUTPUT_DATASET_PATH,exist_ok=True)
    grouped_graphs = {}
    graph_database = {}
    logs = []
    success_count = 0
    group_start = start_timer()

    print(f"{'Input Directory':<30}: {INPUT_DIR}")
    print(f"{'Total Files Found':<30}: {len(all_pkl_files)}")

    for file_path in tqdm(all_pkl_files, desc="Consolidating Dataset", unit="file"):
        filename = os.path.basename(file_path)
        # 1. Load Data
        G, meta, status, msg = load_single_graph_data(file_path)
        nim, modul, file = extract_assignment_name(filename)
        
        if status == "SUCCESS" and G is not None:
            try:
                # 2. Normalisasi Graf untuk Karateclub
                G_norm = normalize_graph_for_karateclub(G)
                graph_key = (
                    f"{nim}_"
                    f"{modul}_"
                    f"{file}"
                )
                graph_database[graph_key] = G_norm
                if is_eval:
                    # Seluruh graph evaluasi (asli, type1, type2, type3)
                    # digabung ke dalam satu dataset Graph2Vec
                    key = ("eval_dataset", "all_graphs") 
                else:
                    # Jika data mahasiswa nyata operasional, kelompokkan normal per modul & file
                    key = (modul, file)
                    
                if key not in grouped_graphs:
                    grouped_graphs[key] = {
                        "graphs": [],
                        "metadata": []
                    }

                grouped_graphs[key]["graphs"].append(G_norm)
                grouped_graphs[key]["metadata"].append(meta)
                
                logs.append(save_log(meta, n_nodes=G_norm.number_of_nodes(), n_edges=G_norm.number_of_edges(), status="SUCCESS", message=""))
                success_count += 1
            except Exception as e:
                logs.append(save_log(meta, status="FAILED", message=f"Normalization Error: {str(e)}"))
        else:
            logs.append(save_log(None, status="FAILED", message=f"{filename} | Load Error: {msg}"))

    # 4. Simpan List of Graphs dan Metadata
    if is_eval:
        if not grouped_graphs:
            raise ValueError(
                "Tidak ada graph yang berhasil digroup untuk evaluasi."
            )
        # Hanya ada satu group: ("evaluation", "all")
        data = next(iter(grouped_graphs.values()))
        output_filename = f"eval_{os.path.basename(os.path.dirname(OUTPUT_DATASET_PATH))}.pkl"
        output_path = os.path.join(OUTPUT_DATASET_PATH, output_filename)
        with open(output_path, "wb") as f:
            pickle.dump(data, f)
    else :
        for (modul, file), data in grouped_graphs.items():
            modul_dir = os.path.join(OUTPUT_DATASET_PATH, modul)
            os.makedirs(modul_dir, exist_ok=True)
            # Untuk data operasional mahasiswa nyata, tetap ditulis eceran per modul-file
            output_path = os.path.join(modul_dir, f"{modul}_{file.replace('.py','')}.pkl")
            with open(output_path, "wb") as f:
                pickle.dump(data, f)

    print(f"\nFinal Dataset disimpan di: {OUTPUT_DATASET_PATH}")
    execution_time = save_execution_time(group_start, stage_name, total_mahasiswa=count_students(INPUT_DIR), total_file=count_all_files(OUTPUT_DATASET_PATH)['total_files'], total_size_kb=get_path_size(OUTPUT_DATASET_PATH))

    # SUMMARY & REPORTING
    # ==============================================================================
    REPORT_PATH = report_path
    os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
    df_logs = pd.DataFrame(logs)

    total_files = len(all_pkl_files)
    success_rate = (success_count / total_files * 100) if total_files > 0 else 0

    # Statistik grouping
    total_groups = len(grouped_graphs)
    group_sizes = [len(v["graphs"]) for v in grouped_graphs.values()]

    avg_graph_per_group = sum(group_sizes) / total_groups if total_groups > 0 else 0
    min_graph = min(group_sizes) if group_sizes else 0
    max_graph = max(group_sizes) if group_sizes else 0


    display(HTML(f"<h3> {stage_name} (GROUPED)</h3>"))
    print(f"{'Waktu Eksekusi':<40}: {execution_time} s")
    print(f"{'Total File PKL Input':<40}: {total_files}")
    print(f"{'Berhasil Diproses (SUCCESS)':<40}: {success_count}")
    print(f"{'Gagal (FAILED)':<40}: {total_files - success_count}")
    print(f"{'Success Rate':<40}: {success_rate:.2f}%")
    print("-" * 75)
    print(f"{'Total Group (modul-file)':<40}: {total_groups}")
    print(f"{'Rata-rata graph per group':<40}: {avg_graph_per_group:.2f}")
    print(f"{'Minimum graph per group':<40}: {min_graph}")
    print(f"{'Maximum graph per group':<40}: {max_graph}")
    print("-" * 75)

    # VALIDASI GROUP
    valid_groups = sum(1 for v in grouped_graphs.values() if len(v["graphs"]) >= 2)
    invalid_groups = total_groups - valid_groups
    print(f"{'Group valid (≥2 graph)':<40}: {valid_groups}")
    print(f"{'Group tidak valid (<2 graph)':<40}: {invalid_groups}")
    print("-" * 75)
    print(f"{'Dataset Output Directory':<40}: {OUTPUT_DATASET_PATH}")

    # Preview
    print("\nPreview Group (5 pertama):")
    preview_data = []
    for i, ((modul, file), v) in enumerate(grouped_graphs.items()):
        if i >= 5:
            break
        preview_data.append({
            "modul": modul,
            "file": file,
            "jumlah_graph": len(v["graphs"])
        })

    df_preview = pd.DataFrame(preview_data)
    display(df_preview)

    # SAVE TO EXCEL (LOG + VALIDATION)
    # Group summary
    group_records = []
    for (modul, file), data in grouped_graphs.items():
        n_graph = len(data["graphs"])
        group_records.append({
            "modul": modul,
            "file": file,
            "jumlah_graph": n_graph,
            "status_group": "VALID" if n_graph >= 2 else "INVALID"
        })
    df_group = pd.DataFrame(group_records)
    # Metadata detail
    meta_records = []
    for (modul, file), data in grouped_graphs.items():
        for m in data["metadata"]:
            meta_records.append({
                "modul": modul,
                "file": file,
                "nim": m.get("nim"),
                "filename": m.get("file")
            })

    df_meta = pd.DataFrame(meta_records)

    # Validation summary
    df_summary = pd.DataFrame([{
        "total_files": total_files,
        "success": success_count,
        "failed": total_files - success_count,
        "success_rate (%)": round(success_rate, 2),
        "total_groups": total_groups,
        "valid_groups": valid_groups,
        "invalid_groups": invalid_groups,
        "avg_graph_per_group": round(avg_graph_per_group, 2),
        "min_graph": min_graph,
        "max_graph": max_graph
    }])

    # Save Excel
    with pd.ExcelWriter(REPORT_PATH, engine="openpyxl") as writer:
        df_logs.to_excel(writer, sheet_name="logs", index=False)
        df_group.to_excel(writer, sheet_name="validation_group", index=False)
        df_meta.to_excel(writer, sheet_name="metadata", index=False)
        df_summary.to_excel(writer, sheet_name="summary", index=False)

        # Auto width
        for ws in writer.sheets.values():
            for col in ws.columns:
                max_len = 0
                col_letter = col[0].column_letter   
                for cell in col:
                    if cell.value:
                        max_len = max(max_len, len(str(cell.value)))
                ws.column_dimensions[col_letter].width = min(max_len + 2, 50)

    print(f"\n📊 Report Excel disimpan di: {REPORT_PATH}")
    print("\n" + "=" * 75)
    
timestamp_finish = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada: {timestamp_finish}")

Diproses pada: 2026-06-11 22:53:55


In [21]:
print("="*60)
print(f"{' LIST OF GRAPHS PRAKTIKUM':^60}")
print("-"*60)
INPUT_DIR = DIRS['GRAPH_P']
OUTPUT_DIR = DIRS['INPUT_GRAPH_P']
REPORT = RESULTS['LIST_GRAPH_P']

build_list_of_graph(INPUT_DIR, OUTPUT_DIR, REPORT, "9_Group_of_Graphs_praktikum")
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

                  LIST OF GRAPHS PRAKTIKUM                  
------------------------------------------------------------
Input Directory               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph_prak
Total Files Found             : 1678


Consolidating Dataset:   0%|          | 0/1678 [00:00<?, ?file/s]


Final Dataset disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\09_Graph2vec_Input_prak

[SELESAI] 9_Group_of_Graphs_praktikum | 81.86 s | 35 file


Waktu Eksekusi                          : 81.86 s
Total File PKL Input                    : 1678
Berhasil Diproses (SUCCESS)             : 1678
Gagal (FAILED)                          : 0
Success Rate                            : 100.00%
---------------------------------------------------------------------------
Total Group (modul-file)                : 35
Rata-rata graph per group               : 47.94
Minimum graph per group                 : 44
Maximum graph per group                 : 53
---------------------------------------------------------------------------
Group valid (≥2 graph)                  : 35
Group tidak valid (<2 graph)            : 0
---------------------------------------------------------------------------
Dataset Output Directory                : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\09_Graph2vec_Input_prak

Preview Group (5 pertama):


,modul,file,jumlah_graph
0,js02,p01,52
1,js02,p02,49
2,js02,p03,49
3,js02,p04,49
4,js03,p01,53



📊 Report Excel disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\09_list_Graph_report_prak.xlsx

Diproses pada 2026-06-11 22:55:18


In [22]:
print("="*60)
print(f"{' LIST OF GRAPHS TUGAS':^60}")
print("-"*60)
INPUT_DIR = DIRS['GRAPH_T']
OUTPUT_DIR = DIRS['INPUT_GRAPH_T']
REPORT = RESULTS['LIST_GRAPH_T']

build_list_of_graph(INPUT_DIR, OUTPUT_DIR, REPORT, "9_Group_of_Graphs_tugas")
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

                    LIST OF GRAPHS TUGAS                    
------------------------------------------------------------
Input Directory               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph_tgs
Total Files Found             : 566


Consolidating Dataset:   0%|          | 0/566 [00:00<?, ?file/s]


Final Dataset disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\09_Graph2vec_Input_tgs

[SELESAI] 9_Group_of_Graphs_tugas | 30.79 s | 12 file


Waktu Eksekusi                          : 30.79 s
Total File PKL Input                    : 566
Berhasil Diproses (SUCCESS)             : 566
Gagal (FAILED)                          : 0
Success Rate                            : 100.00%
---------------------------------------------------------------------------
Total Group (modul-file)                : 12
Rata-rata graph per group               : 47.17
Minimum graph per group                 : 44
Maximum graph per group                 : 49
---------------------------------------------------------------------------
Group valid (≥2 graph)                  : 12
Group tidak valid (<2 graph)            : 0
---------------------------------------------------------------------------
Dataset Output Directory                : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\09_Graph2vec_Input_tgs

Preview Group (5 pertama):


,modul,file,jumlah_graph
0,js02,tp,49
1,js03,tp,49
2,js04,tp,45
3,js13,tp,49
4,js14,tp,44



📊 Report Excel disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\09_list_Graph_report_tgs.xlsx

Diproses pada 2026-06-11 22:55:49


In [23]:
print("="*60)
print(f"{' LIST OF GRAPHS EVAL PRAKTIKUM':^60}")
print("-"*60)
INPUT_DIR = EVAL_P['GRAPH']
OUTPUT_DIR = EVAL_P['INPUT_GRAPH']
REPORT = RESULTS_EVAL_P['LIST_GRAPH']

build_list_of_graph(INPUT_DIR, OUTPUT_DIR, REPORT, "9_Group_of_Graphs_EVAL_praktikum", True)
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

                LIST OF GRAPHS EVAL PRAKTIKUM               
------------------------------------------------------------
Input Directory               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\08_Graph
Total Files Found             : 2515


Consolidating Dataset:   0%|          | 0/2515 [00:00<?, ?file/s]


Final Dataset disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\09_Graph2vec_Input

[SELESAI] 9_Group_of_Graphs_EVAL_praktikum | 161.15 s | 1 file


Waktu Eksekusi                          : 161.15 s
Total File PKL Input                    : 2515
Berhasil Diproses (SUCCESS)             : 2515
Gagal (FAILED)                          : 0
Success Rate                            : 100.00%
---------------------------------------------------------------------------
Total Group (modul-file)                : 1
Rata-rata graph per group               : 2515.00
Minimum graph per group                 : 2515
Maximum graph per group                 : 2515
---------------------------------------------------------------------------
Group valid (≥2 graph)                  : 1
Group tidak valid (<2 graph)            : 0
---------------------------------------------------------------------------
Dataset Output Directory                : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\09_Graph2vec_Input

Preview Group (5 pertama):


,modul,file,jumlah_graph
0,eval_dataset,all_graphs,2515



📊 Report Excel disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\praktikum\09_list_Graph_report.xlsx

Diproses pada 2026-06-11 22:59:05


In [24]:
print("="*60)
print(f"{' LIST OF GRAPHS EVAL TUGAS':^60}")
print("-"*60)
INPUT_DIR = EVAL_T['GRAPH']
OUTPUT_DIR = EVAL_T['INPUT_GRAPH']
REPORT = RESULTS_EVAL_T['LIST_GRAPH']

build_list_of_graph(INPUT_DIR, OUTPUT_DIR, REPORT, "9_Group_of_Graphs_EVAL_tugas", True)
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

                  LIST OF GRAPHS EVAL TUGAS                 
------------------------------------------------------------
Input Directory               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\08_Graph
Total Files Found             : 850


Consolidating Dataset:   0%|          | 0/850 [00:00<?, ?file/s]


Final Dataset disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\09_Graph2vec_Input

[SELESAI] 9_Group_of_Graphs_EVAL_tugas | 98.85 s | 1 file


Waktu Eksekusi                          : 98.85 s
Total File PKL Input                    : 850
Berhasil Diproses (SUCCESS)             : 850
Gagal (FAILED)                          : 0
Success Rate                            : 100.00%
---------------------------------------------------------------------------
Total Group (modul-file)                : 1
Rata-rata graph per group               : 850.00
Minimum graph per group                 : 850
Maximum graph per group                 : 850
---------------------------------------------------------------------------
Group valid (≥2 graph)                  : 1
Group tidak valid (<2 graph)            : 0
---------------------------------------------------------------------------
Dataset Output Directory                : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\09_Graph2vec_Input

Preview Group (5 pertama):


,modul,file,jumlah_graph
0,eval_dataset,all_graphs,850



📊 Report Excel disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\tugas\09_list_Graph_report.xlsx

Diproses pada 2026-06-11 23:00:47
